<a href="https://colab.research.google.com/github/Srvcl/Servacal.Auxiliar/blob/main/Invoices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Instrucciones
Primero hay que rellenar los conceptos facturables en la excel de **works** que está en el directorio de Drive en este enlace: https://drive.google.com/drive/folders/1F5ltjVukQHw3KGFc60-mQuDsn2b1khDR?usp=share_link.

Posteriormente, hay que verificar si la empresa esta en la excel **clientes** del mismo directorio.

Finalmente, hay que descargar en CSV, las dos excel de **works** y de **clientes** para luego subirlas al entorno virtual y ejecutar los **Programas** según el tipo de documento que se quiera elaborar.

## Factura IVA (10) Only

In [ ]:
import pandas as pd
import datetime
!pip install reportlab
import reportlab
from pathlib import Path
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

class InvoiceGenerator:
    def __init__(self, clients_csv_path, works_csv_path, tax_rate=0.21):
        """
        Initialize the Invoice Generator

        Args:
            clients_csv_path (str): Path to CSV file containing client data
            works_csv_path (str): Path to CSV file containing work/service data
            tax_rate (float): Tax rate (default 21% = 0.21)
        """
        self.clients_csv_path = clients_csv_path
        self.works_csv_path = works_csv_path
        self.tax_rate = tax_rate

        # Load data
        self.clients_df = self.load_clients_data()
        self.works_df = self.load_works_data()

    def load_clients_data(self):
        """Load client data from CSV file"""
        try:
            df = pd.read_csv(self.clients_csv_path)
            # Clean column names (remove whitespace)
            df.columns = df.columns.str.strip()
            print(f"Loaded {len(df)} clients from {self.clients_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Client file '{self.clients_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading client data: {e}")
            return pd.DataFrame()

    def load_works_data(self):
        """Load works/services data from CSV file"""
        try:
            df = pd.read_csv(self.works_csv_path)
            # Clean column names (remove whitespace)
            df.columns = df.columns.str.strip()
            # Convert date column to datetime if it exists
            if 'date' in df.columns.str.lower():
                date_col = [col for col in df.columns if 'date' in col.lower()][0]
                df[date_col] = pd.to_datetime(df[date_col])
            print(f"Loaded {len(df)} work records from {self.works_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Works file '{self.works_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading works data: {e}")
            return pd.DataFrame()

    def get_client_data(self, client_id):
        """
        Retrieve client data by ID

        Args:
            client_id: The client ID to search for

        Returns:
            dict: Client data or None if not found
        """
        # Try different possible column names for client ID
        id_columns = ['id', 'client_id', 'ID', 'Client_ID', 'id_number', 'ID_Number', 'Client_KEY']
        client_id_col = None

        for col in id_columns:
            if col in self.clients_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in clients data")
            return None

        # Debug: Print available client IDs and their types
        print(f"Debug: Looking for client ID '{client_id}' (type: {type(client_id)})")
        print(f"Debug: Available client IDs: {self.clients_df[client_id_col].tolist()}")
        print(f"Debug: Client ID types: {[type(x) for x in self.clients_df[client_id_col].tolist()]}")

        # Convert both to string for comparison to handle type mismatches
        client_id_str = str(client_id)
        client_row = self.clients_df[self.clients_df[client_id_col].astype(str) == client_id_str]

        if client_row.empty:
            print(f"Error: Client with ID '{client_id}' not found")
            return None

        return client_row.iloc[0].to_dict()

    def get_client_works(self, client_id):
        """
        Retrieve all works for a specific client

        Args:
            client_id: The client ID to search for

        Returns:
            DataFrame: Works data for the client
        """
        # Try different possible column names for client ID in works data
        id_columns = ['client_id', 'Client_ID', 'id', 'ID', 'client', 'Client', 'Client_KEY']
        client_id_col = None

        for col in id_columns:
            if col in self.works_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in works data")
            return pd.DataFrame()

        client_works = self.works_df[self.works_df[client_id_col] == client_id].copy()
        return client_works

    def calculate_invoice_totals(self, works_df):
        """
        Calculate invoice totals from works data

        Args:
            works_df (DataFrame): Works data for a client

        Returns:
            dict: Dictionary containing subtotal, tax, and total amounts
        """
        # Try different possible column names for amounts
        amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Importe']
        amount_col = None

        for col in amount_columns:
            if col in works_df.columns:
                amount_col = col
                break

        if amount_col is None:
            print("Error: Could not find amount column in works data")
            return {'subtotal': 0, 'tax': 0, 'total': 0}

        # Calculate subtotal
        subtotal = works_df[amount_col].sum()

        # Calculate tax
        tax_amount = subtotal * self.tax_rate

        # Calculate total
        total = subtotal + tax_amount

        return {
            'subtotal': round(subtotal, 2),
            'tax': round(tax_amount, 2),
            'total': round(total, 2)
        }

    def generate_invoice(self, client_id, invoice_number=None, invoice_date=None):
        """
        Generate a complete invoice for a client

        Args:
            client_id: The client ID
            invoice_number (str): Invoice number (auto-generated if None)
            invoice_date (str): Invoice date (today if None)

        Returns:
            dict: Complete invoice data
        """
        # Get client data
        client_data = self.get_client_data(client_id)
        if client_data is None:
            return None

        # Get client works
        client_works = self.get_client_works(client_id)
        if client_works.empty:
            print(f"Warning: No works found for client ID '{client_id}'")

        # Calculate totals
        totals = self.calculate_invoice_totals(client_works)

        # Generate invoice number if not provided
        invoice_number = input("Enter invoice number (without'/'!!!): ")
        if invoice_number is None:
            #invoice_number = f"INV-{datetime.datetime.now().strftime('%Y%m%d')}-{client_id}"
        	invoice_number = f"A2025_754"

        invoice_date = input("Enter invoice date: ")

        # Set invoice date if not provided
        if invoice_date is None:
            #invoice_date = datetime.datetime.now().strftime('%Y-%m-%d')
        	invoice_date = '2025-11-11'
        # Create invoice dictionary
        invoice = {
            'invoice_number': invoice_number,
            'invoice_date': invoice_date,
            'client_data': client_data,
            'works': client_works.to_dict('records'),
            'subtotal': totals['subtotal'],
            'tax_rate': self.tax_rate * 100,  # Convert to percentage
            'tax_amount': totals['tax'],
            'total_amount': totals['total']
        }

        return invoice

    def print_invoice(self, invoice):
        """Print invoice in a formatted way"""
        if invoice is None:
            print("No invoice to print")
            return

        print("=" * 92)
        print(f"INVOICE #{invoice['invoice_number']}")
        print("=" * 92)
        print(f"Date: {invoice['invoice_date']}")
        print()

        # Client information
        print("Facturado a:")
        client = invoice['client_data']

        # Try to get client name from different possible columns
        name_columns = ['name', 'company_name', 'client_name', 'Name', 'Company_Name', 'Client_Name']
        client_name = "Unknown Client"
        for col in name_columns:
            if col in client and pd.notna(client[col]):
                client_name = client[col]
                break

        print(f"  {client_name}")

        # Print address if available
        address_columns = ['address', 'Address', 'street', 'Street']
        for col in address_columns:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        # Print zip code if available
        zip_columns = ['zip_code', 'zip', 'postal_code', 'Zip_Code', 'ZIP', 'Postal_Code']
        for col in zip_columns:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        print()

        # Works/Services
        print("CONCEPTOS-TRABAJOS:")
        print("-" * 92)
        print(f"{'Description':<70} {'Date':<12} {'Amount':<10}")
        print("-" * 92)

        for work in invoice['works']:
            # Get description
            desc_columns = ['description', 'concept', 'service', 'Description', 'Concept', 'Service']
            description = "Service"
            for col in desc_columns:
                if col in work and pd.notna(work[col]):
                    # Print the full description in a separate line for readability
                    description = str(work[col])
                    break

            # Get date
            date_columns = ['date', 'Date', 'work_date', 'Work_Date']
            work_date = "N/A"
            for col in date_columns:
                if col in work and pd.notna(work[col]):
                    if isinstance(work[col], str):
                        work_date = work[col][:10]
                    else:
                        work_date = str(work[col])[:10]
                    break

            # Get amount
            amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Price']
            amount = 0
            for col in amount_columns:
                if col in work and pd.notna(work[col]):
                    amount = work[col]
                    break

            # Print full description, then date and amount on a new line
            print(f"  Description: {description}")
            print(f"  Date: {work_date:<12} Amount: Eur {amount:<9.2f}")
            print() # Add a blank line for readability

        print("-" * 92)
        print(f"{'Subtotal:':<79} Eur {invoice['subtotal']:>8.2f}")
        print(f"{'Tax (' + str(invoice['tax_rate']) + '%):':<79} Eur {invoice['tax_amount']:>8.2f}")
        print("=" * 92)
        print(f"{'TOTAL:':<79} Eur {invoice['total_amount']:>8.2f}")
        print("=" * 92)

    def save_invoice_to_file(self, invoice, filename=None):
        """Save invoice to a text file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            filename = f"invoice_{invoice['invoice_number']}.txt"

        # Redirect print output to file
        import sys
        original_stdout = sys.stdout

        try:
            with open(filename, 'w') as f:
                sys.stdout = f
                self.print_invoice(invoice)
            sys.stdout = original_stdout
            print(f"Invoice saved to {filename}")
        except Exception as e:
            sys.stdout = original_stdout
            print(f"Error saving invoice: {e}")

    def save_invoice_to_pdf(self, invoice, filename=None):
        """Save invoice to a PDF file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            filename = f"invoice_{invoice['invoice_number']}.pdf"

        try:
            # Create PDF document
            doc = SimpleDocTemplate(filename, pagesize=A4,
                                    rightMargin=72, leftMargin=72,
                                    topMargin=72, bottomMargin=72)

            # Get styles
            styles = getSampleStyleSheet()

            # Create custom styles
            title_style = ParagraphStyle(
                'InvoiceTitle',
                parent=styles['Heading1'],
                fontSize=24,
                spaceAfter=30,
                textColor=colors.darkblue,
                alignment=TA_CENTER,
                fontName='Helvetica-Bold'
            )

            header_style = ParagraphStyle(
                'InvoiceHeader',
                parent=styles['Heading2'],
                fontSize=14,
                spaceAfter=12,
                spaceBefore=20,
                textColor=colors.darkblue,
                fontName='Helvetica-Bold'
            )

            normal_style = ParagraphStyle(
                'InvoiceNormal',
                parent=styles['Normal'],
                fontSize=9,
                spaceAfter=6,
                leading=14,
                fontName='Helvetica'
            )

            client_style = ParagraphStyle(
                'ClientInfo',
                parent=styles['Normal'],
                fontSize=11,
                spaceAfter=6,
                leading=16,
                fontName='Helvetica-Bold'
            )

            # Build PDF content
            story = []

            # Invoice title
            story.append(Paragraph("Factura", title_style))
            story.append(Paragraph("Servacal_SPV Servicios Veterinarios", normal_style))
            story.append(Paragraph("Rafael Vazquez Caldito. 08847362K", normal_style))
            story.append(Paragraph("Enrique Segura Otano, 5. 7A.", normal_style))
            story.append(Paragraph("06004. Badajoz (Badajoz).", normal_style))
            story.append(Spacer(1, 10))

            # Invoice number and date
            story.append(Paragraph(f"<b>Factura. N_#:</b> {str(invoice['invoice_number'])}", normal_style))
            story.append(Paragraph(f"<b>Fecha:</b> {str(invoice['invoice_date'])}", normal_style))
            story.append(Spacer(1, 10))

            # Client information
            story.append(Paragraph("Facturado a:", header_style))
            client = invoice['client_data']

            # Get client name
            name_columns = ['name', 'company_name', 'client_name', 'Name', 'Company_Name', 'Client_Name']
            client_name = "Unknown Client"
            for col in name_columns:
                if col in client and pd.notna(client[col]):
                    client_name = client[col]
                    break

            story.append(Paragraph(str(client_name), client_style))

            # Get address
            address_columns = ['address', 'Address', 'street', 'Street']
            for col in address_columns:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            # Get zip code
            zip_columns = ['zip_code', 'zip', 'postal_code', 'Zip_Code', 'ZIP', 'Postal_Code']
            for col in zip_columns:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            story.append(Spacer(1, 10))

            # Services table
            story.append(Paragraph("SERVICES:", header_style))
            story.append(Spacer(1, 10))

            # Prepare table data
            table_data = [['Description', 'Date', 'Amount']]

            for work in invoice['works']:
                # Get description (no truncation)
                desc_columns = ['description', 'concept', 'service', 'Description', 'Concept', 'Service']
                description = "Service"
                for col in desc_columns:
                    if col in work and pd.notna(work[col]):
                        description = str(work[col])
                        break

                # Get date
                date_columns = ['date', 'Date', 'work_date', 'Work_Date']
                work_date = "N/A"
                for col in date_columns:
                    if col in work and pd.notna(work[col]):
                        work_date = str(work[col])[:10]
                        break

                # Get amount
                amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Price']
                amount = 0
                for col in amount_columns:
                    if col in work and pd.notna(work[col]):
                        amount = work[col]
                        break

                # The fix is here: wrap the description in a Paragraph object
                table_data.append([Paragraph(description, normal_style), work_date, f"Eur {amount:.2f}"])

            # Add totals row
            table_data.append(['', '', ''])
            table_data.append(['Subtotal:', '', f"Eur {invoice['subtotal']:.2f}"])
            table_data.append([f'Tax ({invoice["tax_rate"]:.1f}%):', '', f"Eur {invoice['tax_amount']:.2f}"])
            table_data.append(['TOTAL:', '', f"Eur {invoice['total_amount']:.2f}"])

            # Create table with flexible width for description column
            # The '*' in colWidths allows the first column to expand to fit the content,
            # ensuring all text is displayed.
            table = Table(table_data, colWidths=['*', 1.5*inch, 1*inch])
            table.setStyle(TableStyle([
                # Header row
                ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('ALIGN', (2, 0), (2, -1), 'RIGHT'),  # Amount column right-aligned
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, 0), 12),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),

                # Data rows
                ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
                ('FONTSIZE', (0, 1), (-1, -1), 10),
                ('GRID', (0, 0), (-1, -4), 1, colors.black),  # Grid for data rows
                ('LINEBELOW', (0, -4), (-1, -4), 2, colors.black),  # Line above totals

                # Totals rows
                ('FONTNAME', (0, -3), (-1, -1), 'Helvetica-Bold'),
                ('FONTSIZE', (0, -3), (-1, -1), 11),
                ('LINEABOVE', (0, -1), (-1, -1), 2, colors.black),  # Line above final total
                ('BACKGROUND', (0, -1), (-1, -1), colors.lightgrey),  # Highlight total row
            ]))

            story.append(table)
            story.append(Spacer(1, 30))

            # Footer
            story.append(Paragraph("Cuenta Corriente de abono, numero:", normal_style))
            story.append(Paragraph("ES71 1583 0001 1490 4748 3875", client_style))

            # Build PDF
            doc.build(story)
            print(f"Invoice PDF saved to {filename}")

        except Exception as e:
            print(f"Error saving invoice to PDF: {e}")


# Example usage and demo
def main():
    """
    Example usage of the InvoiceGenerator

    Expected CSV file structures:

    clients.csv:
    - id, name (or company_name), address, zip_code, etc.

    works.csv:
    - client_id, date, description (or concept), amount (or import), etc.
    """

    # Initialize the invoice generator
    # You'll need to replace these paths with your actual CSV file paths
    generator = InvoiceGenerator(
        clients_csv_path="clients.csv",
        works_csv_path="works.csv",
        tax_rate=0.10  # 10% tax rate (adjust as needed)
    )

    # Example: Generate invoice for client with ID 5
    client_id = int(input("Enter ID_client: ")) # Replace with actual client ID

    # Generate the invoice
    invoice = generator.generate_invoice(client_id)

    if invoice:
        # Print the invoice to console
        generator.print_invoice(invoice)

        # Save invoice to text file
        generator.save_invoice_to_file(invoice)

        # Save invoice to PDF file
        generator.save_invoice_to_pdf(invoice)
    else:
        print("Failed to generate invoice")

if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.6 MB/s eta 0:00:00
Error: Client file 'clients.csv' not found
Error: Works file 'works.csv' not found
Enter ID_client: 20
Error: Could not find client ID column in clients data
Failed to generate invoice


## Factura IRPF e IVA (10 or 21)

Ajustar IVA en *linea* 590 aproximadamente según conveniencia

Hay que establecer en **Google Sheet** el **formato automatico** para los precios de los conceptos

In [ ]:
import datetime
import pandas as pd
!pip install reportlab
import reportlab
from pathlib import Path
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

class InvoiceGenerator:
    def __init__(self, clients_csv_path, works_csv_path, tax_rate=0.21, irpf_rate=0.15):
        """
        Initialize the Invoice Generator

        Args:
            clients_csv_path (str): Path to CSV file containing client data
            works_csv_path (str): Path to CSV file containing work/service data
            tax_rate (float): Standard tax rate (e.g., VAT, default 21% = 0.21)
            irpf_rate (float): IRPF withholding rate (default 15% = 0.15)
        """
        self.clients_csv_path = clients_csv_path
        self.works_csv_path = works_csv_path
        self.tax_rate = tax_rate
        self.irpf_rate = irpf_rate # New IRPF rate attribute

        # Load data
        self.clients_df = self.load_clients_data()
        self.works_df = self.load_works_data()

    def load_clients_data(self):
        """Load client data from CSV file"""
        try:
            df = pd.read_csv(self.clients_csv_path)
            # Clean column names (remove whitespace)
            df.columns = df.columns.str.strip()
            print(f"Loaded {len(df)} clients from {self.clients_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Client file '{self.clients_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading client data: {e}")
            return pd.DataFrame()

    def load_works_data(self):
        """Load works/services data from CSV file"""
        try:
            df = pd.read_csv(self.works_csv_path)
            # Clean column names (remove whitespace)
            df.columns = df.columns.str.strip()
            # Convert date column to datetime if it exists
            if 'date' in df.columns.str.lower():
                date_col = [col for col in df.columns if 'date' in col.lower()][0]
                df[date_col] = pd.to_datetime(df[date_col])
            print(f"Loaded {len(df)} work records from {self.works_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Works file '{self.works_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading works data: {e}")
            return pd.DataFrame()

    def get_client_data(self, client_id):
        """
        Retrieve client data by ID

        Args:
            client_id: The client ID to search for

        Returns:
            dict: Client data or None if not found
        """
        # Try different possible column names for client ID
        id_columns = ['id', 'client_id', 'ID', 'Client_ID', 'id_number', 'ID_Number', 'Client_KEY']
        client_id_col = None

        for col in id_columns:
            if col in self.clients_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in clients data")
            return None

        # Debug: Print available client IDs and their types
        print(f"Debug: Looking for client ID '{client_id}' (type: {type(client_id)})")
        print(f"Debug: Available client IDs: {self.clients_df[client_id_col].tolist()}")
        print(f"Debug: Client ID types: {[type(x) for x in self.clients_df[client_id_col].tolist()]}")

        # Convert both to string for comparison to handle type mismatches
        client_id_str = str(client_id)
        client_row = self.clients_df[self.clients_df[client_id_col].astype(str) == client_id_str]

        if client_row.empty:
            print(f"Error: Client with ID '{client_id}' not found")
            return None

        return client_row.iloc[0].to_dict()

    def get_client_works(self, client_id):
        """
        Retrieve all works for a specific client

        Args:
            client_id: The client ID to search for

        Returns:
            DataFrame: Works data for the client
        """
        # Try different possible column names for client ID in works data
        id_columns = ['client_id', 'Client_ID', 'id', 'ID', 'client', 'Client', 'Client_KEY']
        client_id_col = None

        for col in id_columns:
            if col in self.works_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in works data")
            return pd.DataFrame()

        # NOTE: Type conversion is typically handled by pandas when joining, but
        # explicitly match the type of the `client_id` for accurate filtering.
        client_works = self.works_df[self.works_df[client_id_col].astype(str) == str(client_id)].copy()
        return client_works

    def calculate_invoice_totals(self, works_df):
        """
        Calculate invoice totals from works data, including IRPF (added to total as per request).

        Args:
            works_df (DataFrame): Works data for a client

        Returns:
            dict: Dictionary containing subtotal, tax, IRPF, and total amounts
        """
        # Try different possible column names for amounts
        amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Importe']
        amount_col = None

        for col in amount_columns:
            if col in works_df.columns:
                amount_col = col
                break

        if amount_col is None:
            print("Error: Could not find amount column in works data")
            return {'subtotal': 0, 'tax': 0, 'irpf': 0, 'total': 0}

        # Calculate subtotal (Base Imponible)
        subtotal = works_df[amount_col].sum()

        # Calculate standard tax (e.g., VAT/IVA)
        tax_amount = subtotal * self.tax_rate

        # Calculate IRPF (Retention/Withholding)
        irpf_amount = subtotal * self.irpf_rate

        # Calculate total as: Subtotal + VAT + IRPF (Following the user's explicit request to ADD IRPF)
        total = subtotal + tax_amount - irpf_amount

        return {
            'subtotal': round(subtotal, 2),
            'tax': round(tax_amount, 2),
            'irpf': round(irpf_amount, 2), # New IRPF amount
            'total': round(total, 2)
        }

    def generate_invoice(self, client_id, invoice_number=None, invoice_date=None):
        """
        Generate a complete invoice for a client

        Args:
            client_id: The client ID
            invoice_number (str): Invoice number (auto-generated if None)
            invoice_date (str): Invoice date (today if None)

        Returns:
            dict: Complete invoice data
        """
        # Get client data
        client_data = self.get_client_data(client_id)
        if client_data is None:
            return None

        # Get client works
        client_works = self.get_client_works(client_id)
        if client_works.empty:
            print(f"Warning: No works found for client ID '{client_id}'")

        # Calculate totals
        totals = self.calculate_invoice_totals(client_works)

        invoice_number = input("Enter invoice number: ")

        # Generate invoice number if not provided
        if invoice_number is None:
            #invoice_number = f"INV-{datetime.datetime.now().strftime('%Y%m%d')}-{client_id}"
            invoice_number = f"A2025_762"
        # Set invoice date if not provided

        invoice_date = input("Enter invoice date: ")

        if invoice_date is None:
            #invoice_date = datetime.datetime.now().strftime('%Y-%m-%d')
            invoice_date = '2026-01-31'

        # Create invoice dictionary
        invoice = {
            'invoice_number': invoice_number,
            'invoice_date': invoice_date,
            'client_data': client_data,
            'works': client_works.to_dict('records'),
            'subtotal': totals['subtotal'],
            'tax_rate': self.tax_rate * 100,  # Convert to percentage
            'tax_amount': totals['tax'],
            'irpf_rate': self.irpf_rate * 100, # New IRPF rate
            'irpf_amount': totals['irpf'],   # New IRPF amount
            'total_amount': totals['total']
        }

        return invoice

    def print_invoice(self, invoice):
        """Print invoice in a formatted way (updated for IRPF)"""
        if invoice is None:
            print("No invoice to print")
            return

        print("=" * 92)
        print(f"INVOICE #{invoice['invoice_number']}")
        print("=" * 92)
        print(f"Date: {invoice['invoice_date']}")
        print()

        # Client information
        print("BILL TO:")
        client = invoice['client_data']

        # Try to get client name from different possible columns
        name_columns = ['name', 'company_name', 'client_name', 'Name', 'Company_Name', 'Client_Name']
        client_name = "Unknown Client"
        for col in name_columns:
            if col in client and pd.notna(client[col]):
                client_name = client[col]
                break

        print(f"  {client_name}")

        # Print address if available
        address_columns = ['address', 'Address', 'street', 'Street']
        for col in address_columns:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        # Print zip code if available
        zip_columns = ['zip_code', 'zip', 'postal_code', 'Zip_Code', 'ZIP', 'Postal_Code']
        for col in zip_columns:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        print()

        # Works/Services
        print("CONCEPTOS-TRABAJOS:")
        print("-" * 92)
        print(f"{'Description':<70} {'Date':<12} {'Amount':<10}")
        print("-" * 92)

        for work in invoice['works']:
            # Get description
            desc_columns = ['description', 'concept', 'service', 'Description', 'Concept', 'Service']
            description = "Service"
            for col in desc_columns:
                if col in work and pd.notna(work[col]):
                    description = str(work[col])
                    break

            # Get date
            date_columns = ['date', 'Date', 'work_date', 'Work_Date']
            work_date = "N/A"
            for col in date_columns:
                if col in work and pd.notna(work[col]):
                    if isinstance(work[col], str):
                        work_date = work[col][:10]
                    else:
                        work_date = str(work[col])[:10]
                    break

            # Get amount
            amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Price']
            amount = 0
            for col in amount_columns:
                if col in work and pd.notna(work[col]):
                    amount = work[col]
                    break

            # Print full description, then date and amount on a new line
            print(f"  Description: {description}")
            print(f"  Date: {work_date:<12} Amount: Eur {amount:<9.2f}")
            print() # Add a blank line for readability

        print("-" * 92)
        print(f"{'Subtotal (Base Imponible):':<79} Eur {invoice['subtotal']:>8.2f}")
        print(f"{'Tax (' + str(invoice['tax_rate']) + '%):':<79} Eur {invoice['tax_amount']:>8.2f}")

        # New IRPF line
        print(f"{'IRPF (' + str(invoice['irpf_rate']) + '%):':<79} Eur {invoice['irpf_amount']:>8.2f}")

        print("=" * 92)
        print(f"{'TOTAL (Including VAT and IRPF):':<79} Eur {invoice['total_amount']:>8.2f}")
        print("=" * 92)

    def save_invoice_to_file(self, invoice, filename=None):
        """Save invoice to a text file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            filename = f"invoice_{invoice['invoice_number']}.txt"

        # Redirect print output to file
        import sys
        original_stdout = sys.stdout

        try:
            with open(filename, 'w') as f:
                sys.stdout = f
                self.print_invoice(invoice)
            sys.stdout = original_stdout
            print(f"Invoice saved to {filename}")
        except Exception as e:
            sys.stdout = original_stdout
            print(f"Error saving invoice: {e}")

    def save_invoice_to_pdf(self, invoice, filename=None):
        """Save invoice to a PDF file (updated for IRPF)"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            filename = f"invoice_{invoice['invoice_number']}.pdf"

        try:
            # Create PDF document
            doc = SimpleDocTemplate(filename, pagesize=A4,
                                            rightMargin=72, leftMargin=72,
                                            topMargin=72, bottomMargin=72)

            # Get styles
            styles = getSampleStyleSheet()

            # Create custom styles
            title_style = ParagraphStyle(
                'InvoiceTitle',
                parent=styles['Heading1'],
                fontSize=24,
                spaceAfter=30,
                textColor=colors.darkblue,
                alignment=TA_CENTER,
                fontName='Helvetica-Bold'
            )

            header_style = ParagraphStyle(
                'InvoiceHeader',
                parent=styles['Heading2'],
                fontSize=14,
                spaceAfter=12,
                spaceBefore=20,
                textColor=colors.darkblue,
                fontName='Helvetica-Bold'
            )

            normal_style = ParagraphStyle(
                'InvoiceNormal',
                parent=styles['Normal'],
                fontSize=9,
                spaceAfter=6,
                leading=14,
                fontName='Helvetica'
            )

            client_style = ParagraphStyle(
                'ClientInfo',
                parent=styles['Normal'],
                fontSize=11,
                spaceAfter=6,
                leading=16,
                fontName='Helvetica-Bold'
            )

            # Build PDF content
            story = []

            # Invoice title
            story.append(Paragraph("Factura", title_style))
            story.append(Paragraph("Servacal_SPV Servicios Veterinarios", normal_style))
            story.append(Paragraph("Rafael Vazquez Caldito. 08847362K", normal_style))
            story.append(Paragraph("Enrique Segura Otano, 5. 7A.", normal_style))
            story.append(Paragraph("06004. Badajoz (Badajoz).", normal_style))
            story.append(Spacer(1, 10))

            # Invoice number and date
            story.append(Paragraph(f"<b>Factura. N_#:</b> {str(invoice['invoice_number'])}", normal_style))
            story.append(Paragraph(f"<b>Fecha:</b> {str(invoice['invoice_date'])}", normal_style))
            story.append(Spacer(1, 10))

            # Client information
            story.append(Paragraph("Facturado a:", header_style))
            client = invoice['client_data']

            # Get client name
            name_columns = ['name', 'company_name', 'client_name', 'Name', 'Company_Name', 'Client_Name']
            client_name = "Unknown Client"
            for col in name_columns:
                if col in client and pd.notna(client[col]):
                    client_name = client[col]
                    break

            story.append(Paragraph(str(client_name), client_style))

            # Get address
            address_columns = ['address', 'Address', 'street', 'Street']
            for col in address_columns:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            # Get zip code
            zip_columns = ['zip_code', 'zip', 'postal_code', 'Zip_Code', 'ZIP', 'Postal_Code']
            for col in zip_columns:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            story.append(Spacer(1, 10))

            # Services table
            story.append(Paragraph("CONCEPTOS-TRABAJOS:", header_style))
            story.append(Spacer(1, 10))

            # Prepare table data
            table_data = [['Descripcion', 'Fecha', 'Importe']]

            for work in invoice['works']:
                # Get description (no truncation)
                desc_columns = ['description', 'concept', 'service', 'Description', 'Concept', 'Service']
                description = "Service"
                for col in desc_columns:
                    if col in work and pd.notna(work[col]):
                        description = str(work[col])
                        break

                # Get date
                date_columns = ['date', 'Date', 'work_date', 'Work_Date']
                work_date = "N/A"
                for col in date_columns:
                    if col in work and pd.notna(work[col]):
                        work_date = str(work[col])[:10]
                        break

                # Get amount
                amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Price']
                amount = 0
                for col in amount_columns:
                    if col in work and pd.notna(work[col]):
                        amount = work[col]
                        break

                # Wrap the description in a Paragraph object for automatic row height adjustment
                table_data.append([Paragraph(description, normal_style), work_date, f"Eur {amount:.2f}"])

            # Calculate where the totals start (4 rows from the end: subtotal, tax, IRPF, total)
            totals_start_row = len(table_data)

            # Add totals rows
            table_data.append(['', '', '']) # Blank separator row
            table_data.append(['Subtotal (Base Imponible):', '', f"Eur {invoice['subtotal']:.2f}"])
            table_data.append([f'Tax ({invoice["tax_rate"]:.1f}% - IVA):', '', f"Eur {invoice['tax_amount']:.2f}"])
            table_data.append([f'IRPF ({invoice["irpf_rate"]:.1f}%):', '', f"Eur {invoice['irpf_amount']:.2f}"]) # New IRPF row
            table_data.append(['TOTAL:', '', f"Eur {invoice['total_amount']:.2f}"])

            # Create table with flexible width for description column
            # The '*' in colWidths allows the first column to expand to fit the content.
            table = Table(table_data, colWidths=['*', 1.5*inch, 1*inch])

            # Determine the indices for total rows relative to the end
            TOTAL_ROW_COUNT = 5 # Blank, Subtotal, Tax, IRPF, TOTAL
            LAST_ROW = -1
            IRPF_ROW = -2
            TAX_ROW = -3
            SUBTOTAL_ROW = -4
            BLANK_ROW = -5

            table.setStyle(TableStyle([
                # Header row
                ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('ALIGN', (2, 0), (2, -1), 'RIGHT'),  # Amount column right-aligned
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, 0), 12),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),

                # Data rows (from row 1 up to the blank row before totals)
                ('FONTNAME', (0, 1), (-1, BLANK_ROW), 'Helvetica'),
                ('FONTSIZE', (0, 1), (-1, BLANK_ROW), 10),
                ('GRID', (0, 0), (-1, BLANK_ROW), 1, colors.black),  # Grid for data rows

                # Totals label formatting
                ('FONTNAME', (0, SUBTOTAL_ROW), (-1, LAST_ROW), 'Helvetica-Bold'),
                ('FONTSIZE', (0, SUBTOTAL_ROW), (-1, LAST_ROW), 11),

                # Line above totals
                ('LINEBELOW', (0, BLANK_ROW), (-1, BLANK_ROW), 2, colors.black),

                # Highlight Total row
                ('LINEABOVE', (0, LAST_ROW), (-1, LAST_ROW), 2, colors.black),
                ('BACKGROUND', (0, LAST_ROW), (-1, LAST_ROW), colors.lightgrey),
            ]))

            story.append(table)
            story.append(Spacer(1, 30))

            # Footer
            story.append(Paragraph("Cuenta Corriente de abono, numero:", normal_style))
            story.append(Paragraph("ES34 3009 0091 8123 2755 9213", client_style))

            # Build PDF
            doc.build(story)
            print(f"Invoice PDF saved to {filename}")

        except Exception as e:
            print(f"Error saving invoice to PDF: {e}")


# Example usage and demo
def main():
    """
    Example usage of the InvoiceGenerator

    This section also creates mock CSV files for demonstration purposes.
    """

    # --- Create Mock CSV Files for Demonstration ---

    # Client Data (clients.csv)
    #clients_data = {
       # 'id': [11, 22],
       # 'name': ['Veterinaria Central S.L.', 'Granja Los Alcores'],
       # 'address': ['Calle del Sol, 12', 'Avenida de la Luna, 5'],
       # 'zip_code': ['28001 Madrid', '06800 Merida'],
       # 'nif': ['B12345678', 'A87654321']
    #}
    #clients_df = pd.DataFrame(clients_data)
    #clients_df.to_csv("clients.csv", index=False)
    #print("Created mock clients.csv")

    # Works Data (works.csv)
    #works_data = {
       # 'client_id': [11, 11, 22],
       # 'date': ['2025-09-01', '2025-09-20', '2025-09-25'],
       # 'description': [
       #     'Servicio de mantenimiento integral de sistemas de control de temperatura y humedad para el trimestre Q3 2025.',
        #    'Asesoramiento tecnico y consultoria especializada sobre la nueva normativa europea de sanidad animal.',
       #     'Suministro y aplicacion de vacunas preventivas contra la fiebre aftosa.'
       # ],
       # 'amount': [500.00, 250.00, 1500.00]
    #}
    #works_df = pd.DataFrame(works_data)
    #works_df.to_csv("works.csv", index=False)
    #print("Created mock works.csv")

    # ---------------------------------------------

    # Initialize the invoice generator
    generator = InvoiceGenerator(
        clients_csv_path="clients.csv",
        works_csv_path="works.csv",
        tax_rate=0.100,  # 21% VAT/IVA OR 10%
        irpf_rate=0.15 # 15% IRPF (Default is 0.15, but explicitly setting it)
    )

    # Example: Generate invoice for client with ID 11
    client_id = int(input("Enter ID_client: ")) # Replace with actual client ID

    # Generate the invoice
    invoice = generator.generate_invoice(client_id)

    if invoice:
        # Print the invoice to console
        generator.print_invoice(invoice)

        # Save invoice to text file
        generator.save_invoice_to_file(invoice)

        # Save invoice to PDF file
        generator.save_invoice_to_pdf(invoice)
    else:
        print("Failed to generate invoice")

if __name__ == "__main__":
    main()

Loaded 10 clients from clients.csv
Loaded 38 work records from works.csv
Enter ID_client: 5
Debug: Looking for client ID '5' (type: <class 'int'>)
Debug: Available client IDs: [2, 3, 4, 11, 12, 6, 5, 17, 16, 20]
Debug: Client ID types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]
Enter invoice number: A2026-779
Enter invoice date: 2026-09-30
INVOICE #A2026-779
Date: 2026-09-30

BILL TO:
  Valdecarras Sociedad Cooperativa. F06011092
  Francisco Pizarro, 7. La Morera. Badajoz. 06176.
  ZIP: 06176

CONCEPTOS-TRABAJOS:
--------------------------------------------------------------------------------------------
Description                                                            Date         Amount    
--------------------------------------------------------------------------------------------
  Description: Consulta numero de receta de inspeccion de Solis, javier nieto. Problema de t

## Facturas 21 IVA Only

In [ ]:
import pandas as pd
import datetime
!pip install reportlab
import reportlab
from pathlib import Path
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

class InvoiceGenerator:
    def __init__(self, clients_csv_path, works_csv_path, tax_rate=0.21):
        """
        Initialize the Invoice Generator

        Args:
            clients_csv_path (str): Path to CSV file containing client data
            works_csv_path (str): Path to CSV file containing work/service data
            tax_rate (float): Tax rate (default 21% = 0.21)
        """
        self.clients_csv_path = clients_csv_path
        self.works_csv_path = works_csv_path
        self.tax_rate = tax_rate

        # Load data
        self.clients_df = self.load_clients_data()
        self.works_df = self.load_works_data()

    def load_clients_data(self):
        """Load client data from CSV file"""
        try:
            df = pd.read_csv(self.clients_csv_path)
            # Clean column names (remove whitespace)
            df.columns = df.columns.str.strip()
            print(f"Loaded {len(df)} clients from {self.clients_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Client file '{self.clients_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading client data: {e}")
            return pd.DataFrame()

    def load_works_data(self):
        """Load works/services data from CSV file"""
        try:
            df = pd.read_csv(self.works_csv_path)
            # Clean column names (remove whitespace)
            df.columns = df.columns.str.strip()
            # Convert date column to datetime if it exists
            if 'date' in df.columns.str.lower():
                date_col = [col for col in df.columns if 'date' in col.lower()][0]
                df[date_col] = pd.to_datetime(df[date_col])
            print(f"Loaded {len(df)} work records from {self.works_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Works file '{self.works_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading works data: {e}")
            return pd.DataFrame()

    def get_client_data(self, client_id):
        """
        Retrieve client data by ID

        Args:
            client_id: The client ID to search for

        Returns:
            dict: Client data or None if not found
        """
        # Try different possible column names for client ID
        id_columns = ['id', 'client_id', 'ID', 'Client_ID', 'id_number', 'ID_Number', 'Client_KEY']
        client_id_col = None

        for col in id_columns:
            if col in self.clients_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in clients data")
            return None

        # Debug: Print available client IDs and their types
        print(f"Debug: Looking for client ID '{client_id}' (type: {type(client_id)})")
        print(f"Debug: Available client IDs: {self.clients_df[client_id_col].tolist()}")
        print(f"Debug: Client ID types: {[type(x) for x in self.clients_df[client_id_col].tolist()]}")

        # Convert both to string for comparison to handle type mismatches
        client_id_str = str(client_id)
        client_row = self.clients_df[self.clients_df[client_id_col].astype(str) == client_id_str]

        if client_row.empty:
            print(f"Error: Client with ID '{client_id}' not found")
            return None

        return client_row.iloc[0].to_dict()

    def get_client_works(self, client_id):
        """
        Retrieve all works for a specific client

        Args:
            client_id: The client ID to search for

        Returns:
            DataFrame: Works data for the client
        """
        # Try different possible column names for client ID in works data
        id_columns = ['client_id', 'Client_ID', 'id', 'ID', 'client', 'Client', 'Client_KEY']
        client_id_col = None

        for col in id_columns:
            if col in self.works_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in works data")
            return pd.DataFrame()

        client_works = self.works_df[self.works_df[client_id_col] == client_id].copy()
        return client_works

    def calculate_invoice_totals(self, works_df):
        """
        Calculate invoice totals from works data

        Args:
            works_df (DataFrame): Works data for a client

        Returns:
            dict: Dictionary containing subtotal, tax, and total amounts
        """
        # Try different possible column names for amounts
        amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Importe']
        amount_col = None

        for col in amount_columns:
            if col in works_df.columns:
                amount_col = col
                break

        if amount_col is None:
            print("Error: Could not find amount column in works data")
            return {'subtotal': 0, 'tax': 0, 'total': 0}

        # Calculate subtotal
        subtotal = works_df[amount_col].sum()

        # Calculate tax
        tax_amount = subtotal * self.tax_rate

        # Calculate total
        total = subtotal + tax_amount

        return {
            'subtotal': round(subtotal, 2),
            'tax': round(tax_amount, 2),
            'total': round(total, 2)
        }

    def generate_invoice(self, client_id, invoice_number=None, invoice_date=None):
        """
        Generate a complete invoice for a client

        Args:
            client_id: The client ID
            invoice_number (str): Invoice number (auto-generated if None)
            invoice_date (str): Invoice date (today if None)

        Returns:
            dict: Complete invoice data
        """
        # Get client data
        client_data = self.get_client_data(client_id)
        if client_data is None:
            return None

        # Get client works
        client_works = self.get_client_works(client_id)
        if client_works.empty:
            print(f"Warning: No works found for client ID '{client_id}'")

        # Calculate totals
        totals = self.calculate_invoice_totals(client_works)

        # Generate invoice number if not provided
        invoice_number = input("Enter invoice number: ")
        if invoice_number is None:
            #invoice_number = f"INV-{datetime.datetime.now().strftime('%Y%m%d')}-{client_id}"
        	invoice_number = f"A2025_754"

        invoice_date = input("Enter invoice date: ")

        # Set invoice date if not provided
        if invoice_date is None:
            #invoice_date = datetime.datetime.now().strftime('%Y-%m-%d')
        	invoice_date = '2025-11-11'
        # Create invoice dictionary
        invoice = {
            'invoice_number': invoice_number,
            'invoice_date': invoice_date,
            'client_data': client_data,
            'works': client_works.to_dict('records'),
            'subtotal': totals['subtotal'],
            'tax_rate': self.tax_rate * 100,  # Convert to percentage
            'tax_amount': totals['tax'],
            'total_amount': totals['total']
        }

        return invoice

    def print_invoice(self, invoice):
        """Print invoice in a formatted way"""
        if invoice is None:
            print("No invoice to print")
            return

        print("=" * 92)
        print(f"INVOICE #{invoice['invoice_number']}")
        print("=" * 92)
        print(f"Date: {invoice['invoice_date']}")
        print()

        # Client information
        print("Facturado a:")
        client = invoice['client_data']

        # Try to get client name from different possible columns
        name_columns = ['name', 'company_name', 'client_name', 'Name', 'Company_Name', 'Client_Name']
        client_name = "Unknown Client"
        for col in name_columns:
            if col in client and pd.notna(client[col]):
                client_name = client[col]
                break

        print(f"  {client_name}")

        # Print address if available
        address_columns = ['address', 'Address', 'street', 'Street']
        for col in address_columns:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        # Print zip code if available
        zip_columns = ['zip_code', 'zip', 'postal_code', 'Zip_Code', 'ZIP', 'Postal_Code']
        for col in zip_columns:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        print()

        # Works/Services
        print("CONCEPTOS-TRABAJOS:")
        print("-" * 92)
        print(f"{'Description':<70} {'Date':<12} {'Amount':<10}")
        print("-" * 92)

        for work in invoice['works']:
            # Get description
            desc_columns = ['description', 'concept', 'service', 'Description', 'Concept', 'Service']
            description = "Service"
            for col in desc_columns:
                if col in work and pd.notna(work[col]):
                    # Print the full description in a separate line for readability
                    description = str(work[col])
                    break

            # Get date
            date_columns = ['date', 'Date', 'work_date', 'Work_Date']
            work_date = "N/A"
            for col in date_columns:
                if col in work and pd.notna(work[col]):
                    if isinstance(work[col], str):
                        work_date = work[col][:10]
                    else:
                        work_date = str(work[col])[:10]
                    break

            # Get amount
            amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Price']
            amount = 0
            for col in amount_columns:
                if col in work and pd.notna(work[col]):
                    amount = work[col]
                    break

            # Print full description, then date and amount on a new line
            print(f"  Description: {description}")
            print(f"  Date: {work_date:<12} Amount: Eur {amount:<9.2f}")
            print() # Add a blank line for readability

        print("-" * 92)
        print(f"{'Subtotal:':<79} Eur {invoice['subtotal']:>8.2f}")
        print(f"{'Tax (' + str(invoice['tax_rate']) + '%):':<79} Eur {invoice['tax_amount']:>8.2f}")
        print("=" * 92)
        print(f"{'TOTAL:':<79} Eur {invoice['total_amount']:>8.2f}")
        print("=" * 92)

    def save_invoice_to_file(self, invoice, filename=None):
        """Save invoice to a text file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            filename = f"invoice_{invoice['invoice_number']}.txt"

        # Redirect print output to file
        import sys
        original_stdout = sys.stdout

        try:
            with open(filename, 'w') as f:
                sys.stdout = f
                self.print_invoice(invoice)
            sys.stdout = original_stdout
            print(f"Invoice saved to {filename}")
        except Exception as e:
            sys.stdout = original_stdout
            print(f"Error saving invoice: {e}")

    def save_invoice_to_pdf(self, invoice, filename=None):
        """Save invoice to a PDF file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            filename = f"invoice_{invoice['invoice_number']}.pdf"

        try:
            # Create PDF document
            doc = SimpleDocTemplate(filename, pagesize=A4,
                                    rightMargin=72, leftMargin=72,
                                    topMargin=72, bottomMargin=72)

            # Get styles
            styles = getSampleStyleSheet()

            # Create custom styles
            title_style = ParagraphStyle(
                'InvoiceTitle',
                parent=styles['Heading1'],
                fontSize=24,
                spaceAfter=30,
                textColor=colors.darkblue,
                alignment=TA_CENTER,
                fontName='Helvetica-Bold'
            )

            header_style = ParagraphStyle(
                'InvoiceHeader',
                parent=styles['Heading2'],
                fontSize=14,
                spaceAfter=12,
                spaceBefore=20,
                textColor=colors.darkblue,
                fontName='Helvetica-Bold'
            )

            normal_style = ParagraphStyle(
                'InvoiceNormal',
                parent=styles['Normal'],
                fontSize=9,
                spaceAfter=6,
                leading=14,
                fontName='Helvetica'
            )

            client_style = ParagraphStyle(
                'ClientInfo',
                parent=styles['Normal'],
                fontSize=11,
                spaceAfter=6,
                leading=16,
                fontName='Helvetica-Bold'
            )

            # Build PDF content
            story = []

            # Invoice title
            story.append(Paragraph("Factura", title_style))
            story.append(Paragraph("Servacal_SPV Servicios Veterinarios", normal_style))
            story.append(Paragraph("Rafael Vazquez Caldito. 08847362K", normal_style))
            story.append(Paragraph("Enrique Segura Otano, 5. 7A.", normal_style))
            story.append(Paragraph("06004. Badajoz (Badajoz).", normal_style))
            story.append(Spacer(1, 10))

            # Invoice number and date
            story.append(Paragraph(f"<b>Factura. N_#:</b> {str(invoice['invoice_number'])}", normal_style))
            story.append(Paragraph(f"<b>Fecha:</b> {str(invoice['invoice_date'])}", normal_style))
            story.append(Spacer(1, 10))

            # Client information
            story.append(Paragraph("Facturado a:", header_style))
            client = invoice['client_data']

            # Get client name
            name_columns = ['name', 'company_name', 'client_name', 'Name', 'Company_Name', 'Client_Name']
            client_name = "Unknown Client"
            for col in name_columns:
                if col in client and pd.notna(client[col]):
                    client_name = client[col]
                    break

            story.append(Paragraph(str(client_name), client_style))

            # Get address
            address_columns = ['address', 'Address', 'street', 'Street']
            for col in address_columns:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            # Get zip code
            zip_columns = ['zip_code', 'zip', 'postal_code', 'Zip_Code', 'ZIP', 'Postal_Code']
            for col in zip_columns:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            story.append(Spacer(1, 10))

            # Services table
            story.append(Paragraph("SERVICES:", header_style))
            story.append(Spacer(1, 10))

            # Prepare table data
            table_data = [['Description', 'Date', 'Amount']]

            for work in invoice['works']:
                # Get description (no truncation)
                desc_columns = ['description', 'concept', 'service', 'Description', 'Concept', 'Service']
                description = "Service"
                for col in desc_columns:
                    if col in work and pd.notna(work[col]):
                        description = str(work[col])
                        break

                # Get date
                date_columns = ['date', 'Date', 'work_date', 'Work_Date']
                work_date = "N/A"
                for col in date_columns:
                    if col in work and pd.notna(work[col]):
                        work_date = str(work[col])[:10]
                        break

                # Get amount
                amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Price']
                amount = 0
                for col in amount_columns:
                    if col in work and pd.notna(work[col]):
                        amount = work[col]
                        break

                # The fix is here: wrap the description in a Paragraph object
                table_data.append([Paragraph(description, normal_style), work_date, f"Eur {amount:.2f}"])

            # Add totals row
            table_data.append(['', '', ''])
            table_data.append(['Subtotal:', '', f"Eur {invoice['subtotal']:.2f}"])
            table_data.append([f'Tax ({invoice["tax_rate"]:.1f}%):', '', f"Eur {invoice['tax_amount']:.2f}"])
            table_data.append(['TOTAL:', '', f"Eur {invoice['total_amount']:.2f}"])

            # Create table with flexible width for description column
            # The '*' in colWidths allows the first column to expand to fit the content,
            # ensuring all text is displayed.
            table = Table(table_data, colWidths=['*', 1.5*inch, 1*inch])
            table.setStyle(TableStyle([
                # Header row
                ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('ALIGN', (2, 0), (2, -1), 'RIGHT'),  # Amount column right-aligned
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, 0), 12),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),

                # Data rows
                ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
                ('FONTSIZE', (0, 1), (-1, -1), 10),
                ('GRID', (0, 0), (-1, -4), 1, colors.black),  # Grid for data rows
                ('LINEBELOW', (0, -4), (-1, -4), 2, colors.black),  # Line above totals

                # Totals rows
                ('FONTNAME', (0, -3), (-1, -1), 'Helvetica-Bold'),
                ('FONTSIZE', (0, -3), (-1, -1), 11),
                ('LINEABOVE', (0, -1), (-1, -1), 2, colors.black),  # Line above final total
                ('BACKGROUND', (0, -1), (-1, -1), colors.lightgrey),  # Highlight total row
            ]))

            story.append(table)
            story.append(Spacer(1, 30))

            # Footer
            story.append(Paragraph("Cuenta Corriente de abono, numero:", normal_style))
            story.append(Paragraph("ES34 3009 0091 8123 2755 9213", client_style))

            # Build PDF
            doc.build(story)
            print(f"Invoice PDF saved to {filename}")

        except Exception as e:
            print(f"Error saving invoice to PDF: {e}")


# Example usage and demo
def main():
    """
    Example usage of the InvoiceGenerator

    Expected CSV file structures:

    clients.csv:
    - id, name (or company_name), address, zip_code, etc.

    works.csv:
    - client_id, date, description (or concept), amount (or import), etc.
    """

    # Initialize the invoice generator
    # You'll need to replace these paths with your actual CSV file paths
    generator = InvoiceGenerator(
        clients_csv_path="clients.csv",
        works_csv_path="works.csv",
        tax_rate=0.21  # 21% tax rate (adjust as needed)
    )

    # Example: Generate invoice for client with ID 5
    client_id = int(input("Enter ID_client: ")) # Replace with actual client ID

    # Generate the invoice
    invoice = generator.generate_invoice(client_id)

    if invoice:
        # Print the invoice to console
        generator.print_invoice(invoice)

        # Save invoice to text file
        generator.save_invoice_to_file(invoice)

        # Save invoice to PDF file
        generator.save_invoice_to_pdf(invoice)
    else:
        print("Failed to generate invoice")

if __name__ == "__main__":
    main()


Loaded 9 clients from clients.csv
Loaded 21 work records from works.csv
Enter ID_client: 5
Debug: Looking for client ID '5' (type: <class 'int'>)
Debug: Available client IDs: [2, 3, 4, 11, 12, 6, 5, 17, 16]
Debug: Client ID types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]
Enter invoice number: A2026-776
Enter invoice date: 2026-07-30
INVOICE #A2026-776
Date: 2026-07-30

Facturado a:
  Valdecarras Sociedad Cooperativa. F06011092
  Francisco Pizarro, 7. La Morera. Badajoz. 06176.
  ZIP: 06176

CONCEPTOS-TRABAJOS:
--------------------------------------------------------------------------------------------
Description                                                            Date         Amount    
--------------------------------------------------------------------------------------------
  Description: Desplazamiento a Valdiguelo para la vacunacion de unos 170 lechones de primera dosis, de mal 

##Albaran del 21-10 % de IVA.
 Ajustar IVA en _linea_ 530 aprox.


In [ ]:
!pip install reportlab
import pandas as pd
import datetime
from pathlib import Path
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

class InvoiceGenerator:
    def __init__(self, clients_csv_path, works_csv_path, tax_rate=0.21):
        """
        Initialize the Invoice Generator

        Args:
            clients_csv_path (str): Path to CSV file containing client data
            works_csv_path (str): Path to CSV file containing work/service data
            tax_rate (float): Tax rate (default 21% = 0.21)
        """
        self.clients_csv_path = clients_csv_path
        self.works_csv_path = works_csv_path
        self.tax_rate = tax_rate

        # Load data
        self.clients_df = self.load_clients_data()
        self.works_df = self.load_works_data()

    def load_clients_data(self):
        """Load client data from CSV file"""
        try:
            df = pd.read_csv(self.clients_csv_path)
            # Clean column names (remove whitespace)
            df.columns = df.columns.str.strip()
            print(f"Loaded {len(df)} clients from {self.clients_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Client file '{self.clients_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading client data: {e}")
            return pd.DataFrame()

    def load_works_data(self):
        """Load works/services data from CSV file"""
        try:
            df = pd.read_csv(self.works_csv_path)
            # Clean column names (remove whitespace)
            df.columns = df.columns.str.strip()
            # Convert date column to datetime if it exists
            if 'date' in df.columns.str.lower():
                date_col = [col for col in df.columns if 'date' in col.lower()][0]
                df[date_col] = pd.to_datetime(df[date_col])
            print(f"Loaded {len(df)} work records from {self.works_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Works file '{self.works_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading works data: {e}")
            return pd.DataFrame()

    def get_client_data(self, client_id):
        """
        Retrieve client data by ID

        Args:
            client_id: The client ID to search for

        Returns:
            dict: Client data or None if not found
        """
        # Try different possible column names for client ID
        id_columns = ['id', 'client_id', 'ID', 'Client_ID', 'id_number', 'ID_Number', 'Client_KEY']
        client_id_col = None

        for col in id_columns:
            if col in self.clients_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in clients data")
            return None

        # Debug: Print available client IDs and their types
        print(f"Debug: Looking for client ID '{client_id}' (type: {type(client_id)})")
        print(f"Debug: Available client IDs: {self.clients_df[client_id_col].tolist()}")
        print(f"Debug: Client ID types: {[type(x) for x in self.clients_df[client_id_col].tolist()]}")

        # Convert both to string for comparison to handle type mismatches
        client_id_str = str(client_id)
        client_row = self.clients_df[self.clients_df[client_id_col].astype(str) == client_id_str]

        if client_row.empty:
            print(f"Error: Client with ID '{client_id}' not found")
            return None

        return client_row.iloc[0].to_dict()

    def get_client_works(self, client_id):
        """
        Retrieve all works for a specific client

        Args:
            client_id: The client ID to search for

        Returns:
            DataFrame: Works data for the client
        """
        # Try different possible column names for client ID in works data
        id_columns = ['client_id', 'Client_ID', 'id', 'ID', 'client', 'Client', 'Client_KEY']
        client_id_col = None

        for col in id_columns:
            if col in self.works_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in works data")
            return pd.DataFrame()

        client_works = self.works_df[self.works_df[client_id_col] == client_id].copy()
        return client_works

    def calculate_invoice_totals(self, works_df):
        """
        Calculate invoice totals from works data

        Args:
            works_df (DataFrame): Works data for a client

        Returns:
            dict: Dictionary containing subtotal, tax, and total amounts
        """
        # Try different possible column names for amounts
        amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Importe']
        amount_col = None

        for col in amount_columns:
            if col in works_df.columns:
                amount_col = col
                break

        if amount_col is None:
            print("Error: Could not find amount column in works data")
            return {'subtotal': 0, 'tax': 0, 'total': 0}

        # Calculate subtotal
        subtotal = works_df[amount_col].sum()

        # Calculate tax
        tax_amount = subtotal * self.tax_rate

        # Calculate total
        total = subtotal + tax_amount

        return {
            'subtotal': round(subtotal, 2),
            'tax': round(tax_amount, 2),
            'total': round(total, 2)
        }

    def generate_invoice(self, client_id, invoice_number=None, invoice_date=None):
        """
        Generate a complete invoice for a client

        Args:
            client_id: The client ID
            invoice_number (str): Invoice number (auto-generated if None)
            invoice_date (str): Invoice date (today if None)

        Returns:
            dict: Complete invoice data
        """
        # Get client data
        client_data = self.get_client_data(client_id)
        if client_data is None:
            return None

        # Get client works
        client_works = self.get_client_works(client_id)
        if client_works.empty:
            print(f"Warning: No works found for client ID '{client_id}'")

        # Calculate totals
        totals = self.calculate_invoice_totals(client_works)

        # Generate invoice number if not provided
        if invoice_number is None:
            #invoice_number = f"INV-{datetime.datetime.now().strftime('%Y%m%d')}-{client_id}"
        	invoice_number = f"1141.30112025"
        # Set invoice date if not provided
        if invoice_date is None:
            #invoice_date = datetime.datetime.now().strftime('%Y-%m-%d')
        	invoice_date = '2025-11-30'
        # Create invoice dictionary
        invoice = {
            'invoice_number': invoice_number,
            'invoice_date': invoice_date,
            'client_data': client_data,
            'works': client_works.to_dict('records'),
            'subtotal': totals['subtotal'],
            'tax_rate': self.tax_rate * 100,  # Convert to percentage
            'tax_amount': totals['tax'],
            'total_amount': totals['total']
        }

        return invoice

    def print_invoice(self, invoice):
        """Print invoice in a formatted way"""
        if invoice is None:
            print("No invoice to print")
            return

        print("=" * 92)
        print(f"INVOICE #{invoice['invoice_number']}")
        print("=" * 92)
        print(f"Date: {invoice['invoice_date']}")
        print()

        # Client information
        print("BILL TO:")
        client = invoice['client_data']

        # Try to get client name from different possible columns
        name_columns = ['name', 'company_name', 'client_name', 'Name', 'Company_Name', 'Client_Name']
        client_name = "Unknown Client"
        for col in name_columns:
            if col in client and pd.notna(client[col]):
                client_name = client[col]
                break

        print(f"  {client_name}")

        # Print address if available
        address_columns = ['address', 'Address', 'street', 'Street']
        for col in address_columns:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        # Print zip code if available
        zip_columns = ['zip_code', 'zip', 'postal_code', 'Zip_Code', 'ZIP', 'Postal_Code']
        for col in zip_columns:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        print()

        # Works/Services
        print("CONCEPTOS-TRABAJOS:")
        print("-" * 92)
        print(f"{'Description':<70} {'Date':<12} {'Amount':<10}")
        print("-" * 92)

        for work in invoice['works']:
            # Get description
            desc_columns = ['description', 'concept', 'service', 'Description', 'Concept', 'Service']
            description = "Service"
            for col in desc_columns:
                if col in work and pd.notna(work[col]):
                    # Print the full description in a separate line for readability
                    description = str(work[col])
                    break

            # Get date
            date_columns = ['date', 'Date', 'work_date', 'Work_Date']
            work_date = "N/A"
            for col in date_columns:
                if col in work and pd.notna(work[col]):
                    if isinstance(work[col], str):
                        work_date = work[col][:10]
                    else:
                        work_date = str(work[col])[:10]
                    break

            # Get amount
            amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Price']
            amount = 0
            for col in amount_columns:
                if col in work and pd.notna(work[col]):
                    amount = work[col]
                    break

            # Print full description, then date and amount on a new line
            print(f"  Description: {description}")
            print(f"  Date: {work_date:<12} Amount: Eur {amount:<9.2f}")
            print() # Add a blank line for readability

        print("-" * 92)
        print(f"{'Subtotal:':<79} Eur {invoice['subtotal']:>8.2f}")
        print(f"{'Tax (' + str(invoice['tax_rate']) + '%):':<79} Eur {invoice['tax_amount']:>8.2f}")
        print("=" * 92)
        print(f"{'TOTAL:':<79} Eur {invoice['total_amount']:>8.2f}")
        print("=" * 92)

    def save_invoice_to_file(self, invoice, filename=None):
        """Save invoice to a text file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            filename = f"invoice_{invoice['invoice_number']}.txt"

        # Redirect print output to file
        import sys
        original_stdout = sys.stdout

        try:
            with open(filename, 'w') as f:
                sys.stdout = f
                self.print_invoice(invoice)
            sys.stdout = original_stdout
            print(f"Invoice saved to {filename}")
        except Exception as e:
            sys.stdout = original_stdout
            print(f"Error saving invoice: {e}")

    def save_invoice_to_pdf(self, invoice, filename=None):
        """Save invoice to a PDF file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            filename = f"invoice_{invoice['invoice_number']}.pdf"

        try:
            # Create PDF document
            doc = SimpleDocTemplate(filename, pagesize=A4,
                                    rightMargin=72, leftMargin=72,
                                    topMargin=72, bottomMargin=72)

            # Get styles
            styles = getSampleStyleSheet()

            # Create custom styles
            title_style = ParagraphStyle(
                'InvoiceTitle',
                parent=styles['Heading1'],
                fontSize=24,
                spaceAfter=30,
                textColor=colors.darkblue,
                alignment=TA_CENTER,
                fontName='Helvetica-Bold'
            )

            header_style = ParagraphStyle(
                'InvoiceHeader',
                parent=styles['Heading2'],
                fontSize=14,
                spaceAfter=12,
                spaceBefore=20,
                textColor=colors.darkblue,
                fontName='Helvetica-Bold'
            )

            normal_style = ParagraphStyle(
                'InvoiceNormal',
                parent=styles['Normal'],
                fontSize=9,
                spaceAfter=6,
                leading=14,
                fontName='Helvetica'
            )

            client_style = ParagraphStyle(
                'ClientInfo',
                parent=styles['Normal'],
                fontSize=11,
                spaceAfter=6,
                leading=16,
                fontName='Helvetica-Bold'
            )

            # Build PDF content
            story = []

            # Invoice title
            story.append(Paragraph("Albaran", title_style))
            story.append(Paragraph("Servacal_SPV Servicios Veterinarios", normal_style))
            story.append(Paragraph("Rafael Vazquez Caldito. 08847362K", normal_style))
            story.append(Paragraph("Enrique Segura Otano, 5. 7A.", normal_style))
            story.append(Paragraph("06004. Badajoz (Badajoz).", normal_style))
            story.append(Spacer(1, 10))

            # Invoice number and date
            story.append(Paragraph(f"<b>Albaran. N_#:</b> {str(invoice['invoice_number'])}", normal_style))
            story.append(Paragraph(f"<b>Fecha:</b> {str(invoice['invoice_date'])}", normal_style))
            story.append(Spacer(1, 10))

            # Client information
            story.append(Paragraph("Cliente de referencia:", header_style))
            client = invoice['client_data']

            # Get client name
            name_columns = ['name', 'company_name', 'client_name', 'Name', 'Company_Name', 'Client_Name']
            client_name = "Unknown Client"
            for col in name_columns:
                if col in client and pd.notna(client[col]):
                    client_name = client[col]
                    break

            story.append(Paragraph(str(client_name), client_style))

            # Get address
            address_columns = ['address', 'Address', 'street', 'Street']
            for col in address_columns:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            # Get zip code
            zip_columns = ['zip_code', 'zip', 'postal_code', 'Zip_Code', 'ZIP', 'Postal_Code']
            for col in zip_columns:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            story.append(Spacer(1, 10))

            # Services table
            story.append(Paragraph("SERVICES:", header_style))
            story.append(Spacer(1, 10))

            # Prepare table data
            table_data = [['Description', 'Date', 'Amount']]

            for work in invoice['works']:
                # Get description (no truncation)
                desc_columns = ['description', 'concept', 'service', 'Description', 'Concept', 'Service']
                description = "Service"
                for col in desc_columns:
                    if col in work and pd.notna(work[col]):
                        description = str(work[col])
                        break

                # Get date
                date_columns = ['date', 'Date', 'work_date', 'Work_Date']
                work_date = "N/A"
                for col in date_columns:
                    if col in work and pd.notna(work[col]):
                        work_date = str(work[col])[:10]
                        break

                # Get amount
                amount_columns = ['amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import', 'Price']
                amount = 0
                for col in amount_columns:
                    if col in work and pd.notna(work[col]):
                        amount = work[col]
                        break

                # The fix is here: wrap the description in a Paragraph object
                table_data.append([Paragraph(description, normal_style), work_date, f"Eur {amount:.2f}"])

            # Add totals row
            table_data.append(['', '', ''])
            table_data.append(['Subtotal:', '', f"Eur {invoice['subtotal']:.2f}"])
            table_data.append([f'Tax ({invoice["tax_rate"]:.1f}%):', '', f"Eur {invoice['tax_amount']:.2f}"])
            table_data.append(['TOTAL:', '', f"Eur {invoice['total_amount']:.2f}"])

            # Create table with flexible width for description column
            # The '*' in colWidths allows the first column to expand to fit the content,
            # ensuring all text is displayed.
            table = Table(table_data, colWidths=['*', 1.5*inch, 1*inch])
            table.setStyle(TableStyle([
                # Header row
                ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('ALIGN', (2, 0), (2, -1), 'RIGHT'),  # Amount column right-aligned
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, 0), 12),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),

                # Data rows
                ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
                ('FONTSIZE', (0, 1), (-1, -1), 10),
                ('GRID', (0, 0), (-1, -4), 1, colors.black),  # Grid for data rows
                ('LINEBELOW', (0, -4), (-1, -4), 2, colors.black),  # Line above totals

                # Totals rows
                ('FONTNAME', (0, -3), (-1, -1), 'Helvetica-Bold'),
                ('FONTSIZE', (0, -3), (-1, -1), 11),
                ('LINEABOVE', (0, -1), (-1, -1), 2, colors.black),  # Line above final total
                ('BACKGROUND', (0, -1), (-1, -1), colors.lightgrey),  # Highlight total row
            ]))

            story.append(table)
            story.append(Spacer(1, 30))

            # Footer
            story.append(Paragraph("Cuenta Corriente de abono, numero:", normal_style))
            story.append(Paragraph("ES34 3009 0091 8123 2755 9213", client_style))

            # Build PDF
            doc.build(story)
            print(f"Invoice PDF saved to {filename}")

        except Exception as e:
            print(f"Error saving invoice to PDF: {e}")


# Example usage and demo
def main():
    """
    Example usage of the InvoiceGenerator

    Expected CSV file structures:

    clients.csv:
    - id, name (or company_name), address, zip_code, etc.

    works.csv:
    - client_id, date, description (or concept), amount (or import), etc.
    """

    # Initialize the invoice generator
    # You'll need to replace these paths with your actual CSV file paths
    generator = InvoiceGenerator(
        clients_csv_path="clients.csv",
        works_csv_path="works.csv",
        tax_rate=0.100  # 21% tax rate (adjust as needed)
    )

    # Example: Generate invoice for client with ID 5
    client_id = 4  # Replace with actual client ID

    # Generate the invoice
    invoice = generator.generate_invoice(client_id)

    if invoice:
        # Print the invoice to console
        generator.print_invoice(invoice)

        # Save invoice to text file
        generator.save_invoice_to_file(invoice)

        # Save invoice to PDF file
        generator.save_invoice_to_pdf(invoice)
    else:
        print("Failed to generate invoice")

if __name__ == "__main__":
    main()


Loaded 9 clients from clients.csv
Loaded 43 work records from works.csv
Debug: Looking for client ID '4' (type: <class 'int'>)
Debug: Available client IDs: [2, 3, 4, 11, 12, 6, 5, 17, 16]
Debug: Client ID types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]
INVOICE #1141.30112025
Date: 2025-11-30

BILL TO:
  Fermin Alejandro Talaverano Rosa
  Finca La Aduana. Valverde de Leganes
  06130

CONCEPTOS-TRABAJOS:
--------------------------------------------------------------------------------------------
Description                                                            Date         Amount    
--------------------------------------------------------------------------------------------
  Description: Realizacion de entrada en Itaca, de los cerdos de bellota cruzados de la aduana de Fermin Alejandro, Talavera Rosa. Acceso a plataforma y geston de documentacion y tramitacion administrativa.
  Date: 202

## NEW Facturas IRPF e IVA (10 or 21)

In [ ]:
import datetime
import pandas as pd
from pathlib import Path
import reportlab
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

class InvoiceGenerator:
    def __init__(self, clients_csv_path, works_csv_path, tax_rate=0.21, irpf_rate=0.15):
        """
        Initialize the Invoice Generator

        Args:
            clients_csv_path (str): Path to CSV file containing client data
            works_csv_path (str): Path to CSV file containing work/service data
            tax_rate (float): Standard tax rate (e.g., VAT, default 21% = 0.21)
            irpf_rate (float): IRPF withholding rate (default 15% = 0.15)
        """
        self.clients_csv_path = clients_csv_path
        self.works_csv_path = works_csv_path
        self.tax_rate = tax_rate
        self.irpf_rate = irpf_rate

        # Load data
        self.clients_df = self.load_clients_data()
        self.works_df = self.load_works_data()

    def load_clients_data(self):
        """Load client data from CSV file"""
        try:
            df = pd.read_csv(self.clients_csv_path)
            df.columns = df.columns.str.strip()
            print(f"Loaded {len(df)} clients from {self.clients_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Client file '{self.clients_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading client data: {e}")
            return pd.DataFrame()

    def load_works_data(self):
        """Load works/services data from CSV file"""
        try:
            df = pd.read_csv(self.works_csv_path)
            df.columns = df.columns.str.strip()
            # Convert date column to datetime if it exists
            date_cols = [col for col in df.columns if 'date' in col.lower()]
            if date_cols:
                df[date_cols[0]] = pd.to_datetime(df[date_cols[0]])
            print(f"Loaded {len(df)} work records from {self.works_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Works file '{self.works_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading works data: {e}")
            return pd.DataFrame()

    def get_client_data(self, client_id):
        """
        Retrieve client data by ID
        """
        id_columns = ['Client_KEY', 'idClients', 'id', 'client_id', 'ID', 'Client_ID', 'id_number', 'ID_Number']
        client_id_col = None

        for col in id_columns:
            if col in self.clients_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in clients data")
            return None

        # Format comparison safely (convert floats like 5.0 to '5')
        def clean_id_str(val):
            try:
                val_float = float(val)
                if val_float.is_integer():
                    return str(int(val_float))
            except (ValueError, TypeError):
                pass
            return str(val).strip()

        target_id_str = clean_id_str(client_id)
        client_row = self.clients_df[self.clients_df[client_id_col].apply(clean_id_str) == target_id_str]

        if client_row.empty:
            print(f"Error: Client with ID '{client_id}' not found")
            return None

        return client_row.iloc[0].to_dict()

    def get_client_works(self, client_id):
        """
        Retrieve all works for a specific client
        """
        id_columns = ['Client_KEY', 'client_id', 'Client_ID', 'id', 'ID', 'client', 'Client']
        client_id_col = None

        for col in id_columns:
            if col in self.works_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in works data")
            return pd.DataFrame()

        def clean_id_str(val):
            try:
                val_float = float(val)
                if val_float.is_integer():
                    return str(int(val_float))
            except (ValueError, TypeError):
                pass
            return str(val).strip()

        target_id_str = clean_id_str(client_id)
        client_works = self.works_df[self.works_df[client_id_col].apply(clean_id_str) == target_id_str].copy()
        return client_works

    def calculate_invoice_totals(self, works_df):
        """
        Calculate invoice totals from works data.
        """
        amount_columns = ['Importe', 'amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import']
        amount_col = None

        for col in amount_columns:
            if col in works_df.columns:
                amount_col = col
                break

        if amount_col is None:
            print("Error: Could not find amount column in works data")
            return {'subtotal': 0, 'tax': 0, 'irpf': 0, 'total': 0}

        subtotal = works_df[amount_col].sum()
        tax_amount = subtotal * self.tax_rate
        irpf_amount = subtotal * self.irpf_rate
        total = subtotal + tax_amount - irpf_amount

        return {
            'subtotal': round(subtotal, 2),
            'tax': round(tax_amount, 2),
            'irpf': round(irpf_amount, 2),
            'total': round(total, 2)
        }

    def generate_invoice(self, client_id, invoice_number=None, invoice_date=None):
        """
        Generate a complete invoice for a client
        """
        client_data = self.get_client_data(client_id)
        if client_data is None:
            return None

        client_works = self.get_client_works(client_id)
        if client_works.empty:
            print(f"Warning: No works found for client ID '{client_id}'")

        totals = self.calculate_invoice_totals(client_works)

        if invoice_number is None:
            inp_num = input("Enter invoice number (or press Enter for auto): ").strip()
            invoice_number = inp_num if inp_num else f"INV-{datetime.datetime.now().strftime('%Y%m%d')}-{client_id}"

        if invoice_date is None:
            inp_date = input("Enter invoice date (YYYY-MM-DD or press Enter for today): ").strip()
            invoice_date = inp_date if inp_date else datetime.datetime.now().strftime('%Y-%m-%d')

        invoice = {
            'invoice_number': invoice_number,
            'invoice_date': invoice_date,
            'client_data': client_data,
            'works': client_works.to_dict('records'),
            'subtotal': totals['subtotal'],
            'tax_rate': self.tax_rate * 100,
            'tax_amount': totals['tax'],
            'irpf_rate': self.irpf_rate * 100,
            'irpf_amount': totals['irpf'],
            'total_amount': totals['total']
        }

        return invoice

    def print_invoice(self, invoice):
        """Print invoice in a formatted way"""
        if invoice is None:
            print("No invoice to print")
            return

        line_sep = "=" * 92
        thin_sep = "-" * 92

        print(line_sep)
        print(f"INVOICE #{invoice['invoice_number']}")
        print(line_sep)
        print(f"Date: {invoice['invoice_date']}\n")

        # Client information
        print("BILL TO:")
        client = invoice['client_data']

        name_cols = ['Name', 'name', 'company_name', 'client_name', 'Company_Name', 'Client_Name']
        client_name = next((client[col] for col in name_cols if col in client and pd.notna(client[col])), "Unknown Client")
        print(f"  {client_name}")

        nif_cols = ['NIF_DNI', 'nif', 'NIF', 'cif', 'CIF']
        for col in nif_cols:
            if col in client and pd.notna(client[col]):
                print(f"  NIF/CIF: {client[col]}")
                break

        addr_cols = ['Adress', 'address', 'Address', 'street', 'Street']
        for col in addr_cols:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        zip_cols = ['ZIP', 'zip_code', 'zip', 'postal_code', 'Zip_Code', 'Postal_Code']
        city_cols = ['City_Town', 'city', 'City']
        zip_val = next((str(int(client[c])) if isinstance(client[c], float) else str(client[c]) for c in zip_cols if c in client and pd.notna(client[c])), "")
        city_val = next((str(client[c]) for c in city_cols if c in client and pd.notna(client[c])), "")

        if zip_val or city_val:
            print(f"  {zip_val} {city_val}".strip())

        print("\nCONCEPTOS-TRABAJOS:")
        print(thin_sep)
        print(f"{'Description':<70} {'Date':<12} {'Amount':<10}")
        print(thin_sep)

        for work in invoice['works']:
            desc_cols = ['Concept', 'description', 'concept', 'service', 'Description', 'Service']
            description = next((str(work[col]) for col in desc_cols if col in work and pd.notna(work[col])), "Service")

            date_cols = ['Date', 'date', 'work_date', 'Work_Date']
            work_date = "N/A"
            for col in date_cols:
                if col in work and pd.notna(work[col]):
                    work_date = str(work[col])[:10]
                    break

            amount_cols = ['Importe', 'amount', 'import', 'price', 'cost', 'value', 'total', 'Amount']
            amount = next((work[col] for col in amount_cols if col in work and pd.notna(work[col])), 0)

            print(f"  Description: {description}")
            print(f"  Date: {work_date:<12} Amount: Eur {amount:<9.2f}\n")

        print(thin_sep)
        print(f"{'Subtotal (Base Imponible):':<79} Eur {invoice['subtotal']:>8.2f}")
        print(f"{'Tax (' + str(invoice['tax_rate']) + '%):':<79} Eur {invoice['tax_amount']:>8.2f}")
        print(f"{'IRPF (' + str(invoice['irpf_rate']) + '%):':<79} Eur {invoice['irpf_amount']:>8.2f}")
        print(line_sep)
        print(f"{'TOTAL (Including VAT and IRPF):':<79} Eur {invoice['total_amount']:>8.2f}")
        print(line_sep)

    def save_invoice_to_file(self, invoice, filename=None):
        """Save invoice to a text file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            safe_num = str(invoice['invoice_number']).replace('/', '_')
            filename = f"invoice_{safe_num}.txt"

        import sys
        original_stdout = sys.stdout

        try:
            with open(filename, 'w', encoding='utf-8') as f:
                sys.stdout = f
                self.print_invoice(invoice)
            sys.stdout = original_stdout
            print(f"Invoice saved to {filename}")
        except Exception as e:
            sys.stdout = original_stdout
            print(f"Error saving invoice: {e}")

    def save_invoice_to_pdf(self, invoice, filename=None):
        """Save invoice to a PDF file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            safe_num = str(invoice['invoice_number']).replace('/', '_')
            filename = f"invoice_{safe_num}.pdf"

        try:
            doc = SimpleDocTemplate(filename, pagesize=A4,
                                    rightMargin=50, leftMargin=50,
                                    topMargin=50, bottomMargin=50)

            styles = getSampleStyleSheet()

            title_style = ParagraphStyle(
                'InvoiceTitle',
                parent=styles['Heading1'],
                fontSize=22,
                spaceAfter=15,
                textColor=colors.darkblue,
                alignment=TA_CENTER,
                fontName='Helvetica-Bold'
            )

            header_style = ParagraphStyle(
                'InvoiceHeader',
                parent=styles['Heading2'],
                fontSize=12,
                spaceAfter=8,
                spaceBefore=14,
                textColor=colors.darkblue,
                fontName='Helvetica-Bold'
            )

            normal_style = ParagraphStyle(
                'InvoiceNormal',
                parent=styles['Normal'],
                fontSize=9,
                spaceAfter=4,
                leading=12,
                fontName='Helvetica'
            )

            client_style = ParagraphStyle(
                'ClientInfo',
                parent=styles['Normal'],
                fontSize=10,
                spaceAfter=4,
                leading=14,
                fontName='Helvetica-Bold'
            )

            story = []

            # Encabezado Emisor
            story.append(Paragraph("Factura", title_style))
            story.append(Paragraph("Servacal_SPV Servicios Veterinarios", normal_style))
            story.append(Paragraph("Rafael Vazquez Caldito. 08847362K", normal_style))
            story.append(Paragraph("Enrique Segura Otano, 5. 7A.", normal_style))
            story.append(Paragraph("06004. Badajoz (Badajoz).", normal_style))
            story.append(Spacer(1, 10))

            # Datos de Factura
            story.append(Paragraph(f"<b>Factura N_#:</b> {str(invoice['invoice_number'])}", normal_style))
            story.append(Paragraph(f"<b>Fecha:</b> {str(invoice['invoice_date'])}", normal_style))
            story.append(Spacer(1, 10))

            # Datos del Cliente
            story.append(Paragraph("Facturado a:", header_style))
            client = invoice['client_data']

            name_cols = ['Name', 'name', 'company_name', 'client_name', 'Company_Name', 'Client_Name']
            client_name = next((client[col] for col in name_cols if col in client and pd.notna(client[col])), "Unknown Client")
            story.append(Paragraph(str(client_name), client_style))

            nif_cols = ['NIF_DNI', 'nif', 'NIF', 'cif', 'CIF']
            for col in nif_cols:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(f"NIF/CIF: {client[col]}", normal_style))
                    break

            addr_cols = ['Adress', 'address', 'Address', 'street', 'Street']
            for col in addr_cols:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            zip_cols = ['ZIP', 'zip_code', 'zip', 'postal_code', 'Zip_Code', 'Postal_Code']
            city_cols = ['City_Town', 'city', 'City']
            zip_val = next((str(int(client[c])) if isinstance(client[c], float) else str(client[c]) for c in zip_cols if c in client and pd.notna(client[c])), "")
            city_val = next((str(client[c]) for c in city_cols if c in client and pd.notna(client[c])), "")

            if zip_val or city_val:
                story.append(Paragraph(f"{zip_val} {city_val}".strip(), normal_style))

            story.append(Spacer(1, 10))

            # Tabla de Conceptos
            story.append(Paragraph("CONCEPTOS-TRABAJOS:", header_style))
            story.append(Spacer(1, 5))

            table_data = [['Descripcion', 'Fecha', 'Importe']]

            for work in invoice['works']:
                desc_cols = ['Concept', 'description', 'concept', 'service', 'Description', 'Service']
                description = next((str(work[col]) for col in desc_cols if col in work and pd.notna(work[col])), "Service")

                date_cols = ['Date', 'date', 'work_date', 'Work_Date']
                work_date = "N/A"
                for col in date_cols:
                    if col in work and pd.notna(work[col]):
                        work_date = str(work[col])[:10]
                        break

                amount_cols = ['Importe', 'amount', 'import', 'price', 'cost', 'value', 'total', 'Amount']
                amount = next((work[col] for col in amount_cols if col in work and pd.notna(work[col])), 0)

                table_data.append([Paragraph(description, normal_style), work_date, f"Eur {amount:.2f}"])

            # Filas de Totales
            table_data.append(['', '', ''])
            table_data.append(['Subtotal (Base Imponible):', '', f"Eur {invoice['subtotal']:.2f}"])
            table_data.append([f'IVA ({invoice["tax_rate"]:.1f}%):', '', f"Eur {invoice['tax_amount']:.2f}"])
            table_data.append([f'IRPF ({invoice["irpf_rate"]:.1f}%):', '', f"Eur {invoice['irpf_amount']:.2f}"])
            table_data.append(['TOTAL:', '', f"Eur {invoice['total_amount']:.2f}"])

            table = Table(table_data, colWidths=['*', 1.2*inch, 1.2*inch])

            LAST_ROW = -1
            SUBTOTAL_ROW = -4
            BLANK_ROW = -5

            table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('ALIGN', (2, 0), (2, -1), 'RIGHT'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, 0), 10),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 8),

                ('FONTNAME', (0, 1), (-1, BLANK_ROW), 'Helvetica'),
                ('FONTSIZE', (0, 1), (-1, BLANK_ROW), 9),
                ('GRID', (0, 0), (-1, BLANK_ROW), 0.5, colors.grey),

                ('FONTNAME', (0, SUBTOTAL_ROW), (-1, LAST_ROW), 'Helvetica-Bold'),
                ('FONTSIZE', (0, SUBTOTAL_ROW), (-1, LAST_ROW), 10),

                ('LINEABOVE', (0, LAST_ROW), (-1, LAST_ROW), 1.5, colors.black),
                ('BACKGROUND', (0, LAST_ROW), (-1, LAST_ROW), colors.lightgrey),
            ]))

            story.append(table)
            story.append(Spacer(1, 20))

            # Pie de página / Datos Bancarios
            story.append(Paragraph("Cuenta Corriente de abono, numero:", normal_style))
            story.append(Paragraph("ES34 3009 0091 8123 2755 9213", client_style))

            doc.build(story)
            print(f"Invoice PDF saved to {filename}")

        except Exception as e:
            print(f"Error saving invoice to PDF: {e}")


def main():
    generator = InvoiceGenerator(
        clients_csv_path="clients.csv",
        works_csv_path="works.csv",
        tax_rate=0.10,   # 21% IVA or 10%
        irpf_rate=0.15   # 15% IRPF
    )

    try:
        client_id_input = input("Enter ID_client: ").strip()
        client_id = int(client_id_input) if client_id_input.isdigit() else client_id_input
    except ValueError:
        client_id = client_id_input

    invoice = generator.generate_invoice(client_id)

    if invoice:
        generator.print_invoice(invoice)
        generator.save_invoice_to_file(invoice)
        generator.save_invoice_to_pdf(invoice)
    else:
        print("Failed to generate invoice")

if __name__ == "__main__":
    main()

Loaded 460 clients from clients.csv
Loaded 2 work records from works.csv
Enter ID_client: 6
Enter invoice number (or press Enter for auto): A2026-782
Enter invoice date (YYYY-MM-DD or press Enter for today): 2026-09-30
INVOICE #A2026-782
Date: 2026-09-30

BILL TO:
  Raizex Agricultura Biosostenible
  NIF/CIF: B02840791
  Avenida de la Innovacion, 2.
6006 Badajoz

CONCEPTOS-TRABAJOS:
--------------------------------------------------------------------------------------------
Description                                                            Date         Amount    
--------------------------------------------------------------------------------------------
  Description: Visita a bonales para la inspeccion de los corderos y de las ovejas que estan pariendo. Tiempo empleado en la observacion de estos animales y del corral con instalaciones. Inspeccion de paridera. Ovinos. Los Bonales. 
  Date: 2026-09-24   Amount: Eur 0.00     

  Description: Cuadre de recetas en libro de medicamento

## NEW Facturas con IRPF del 0.0 y de IVA del 0.10 or .021

In [ ]:
import datetime
import pandas as pd
from pathlib import Path
import reportlab
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

class InvoiceGenerator:
    def __init__(self, clients_csv_path, works_csv_path, tax_rate=0.21, irpf_rate=0.15):
        """
        Initialize the Invoice Generator

        Args:
            clients_csv_path (str): Path to CSV file containing client data
            works_csv_path (str): Path to CSV file containing work/service data
            tax_rate (float): Standard tax rate (e.g., VAT, default 21% = 0.21)
            irpf_rate (float): IRPF withholding rate (default 15% = 0.15)
        """
        self.clients_csv_path = clients_csv_path
        self.works_csv_path = works_csv_path
        self.tax_rate = tax_rate
        self.irpf_rate = irpf_rate

        # Load data
        self.clients_df = self.load_clients_data()
        self.works_df = self.load_works_data()

    def load_clients_data(self):
        """Load client data from CSV file"""
        try:
            df = pd.read_csv(self.clients_csv_path)
            df.columns = df.columns.str.strip()
            print(f"Loaded {len(df)} clients from {self.clients_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Client file '{self.clients_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading client data: {e}")
            return pd.DataFrame()

    def load_works_data(self):
        """Load works/services data from CSV file"""
        try:
            df = pd.read_csv(self.works_csv_path)
            df.columns = df.columns.str.strip()
            # Convert date column to datetime if it exists
            date_cols = [col for col in df.columns if 'date' in col.lower()]
            if date_cols:
                df[date_cols[0]] = pd.to_datetime(df[date_cols[0]])
            print(f"Loaded {len(df)} work records from {self.works_csv_path}")
            return df
        except FileNotFoundError:
            print(f"Error: Works file '{self.works_csv_path}' not found")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error loading works data: {e}")
            return pd.DataFrame()

    def get_client_data(self, client_id):
        """
        Retrieve client data by ID
        """
        id_columns = ['Client_KEY', 'idClients', 'id', 'client_id', 'ID', 'Client_ID', 'id_number', 'ID_Number']
        client_id_col = None

        for col in id_columns:
            if col in self.clients_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in clients data")
            return None

        # Format comparison safely (convert floats like 5.0 to '5')
        def clean_id_str(val):
            try:
                val_float = float(val)
                if val_float.is_integer():
                    return str(int(val_float))
            except (ValueError, TypeError):
                pass
            return str(val).strip()

        target_id_str = clean_id_str(client_id)
        client_row = self.clients_df[self.clients_df[client_id_col].apply(clean_id_str) == target_id_str]

        if client_row.empty:
            print(f"Error: Client with ID '{client_id}' not found")
            return None

        return client_row.iloc[0].to_dict()

    def get_client_works(self, client_id):
        """
        Retrieve all works for a specific client
        """
        id_columns = ['Client_KEY', 'client_id', 'Client_ID', 'id', 'ID', 'client', 'Client']
        client_id_col = None

        for col in id_columns:
            if col in self.works_df.columns:
                client_id_col = col
                break

        if client_id_col is None:
            print("Error: Could not find client ID column in works data")
            return pd.DataFrame()

        def clean_id_str(val):
            try:
                val_float = float(val)
                if val_float.is_integer():
                    return str(int(val_float))
            except (ValueError, TypeError):
                pass
            return str(val).strip()

        target_id_str = clean_id_str(client_id)
        client_works = self.works_df[self.works_df[client_id_col].apply(clean_id_str) == target_id_str].copy()
        return client_works

    def calculate_invoice_totals(self, works_df):
        """
        Calculate invoice totals from works data.
        """
        amount_columns = ['Importe', 'amount', 'import', 'price', 'cost', 'value', 'total', 'Amount', 'Import']
        amount_col = None

        for col in amount_columns:
            if col in works_df.columns:
                amount_col = col
                break

        if amount_col is None:
            print("Error: Could not find amount column in works data")
            return {'subtotal': 0, 'tax': 0, 'irpf': 0, 'total': 0}

        subtotal = works_df[amount_col].sum()
        tax_amount = subtotal * self.tax_rate
        irpf_amount = subtotal * self.irpf_rate
        total = subtotal + tax_amount - irpf_amount

        return {
            'subtotal': round(subtotal, 2),
            'tax': round(tax_amount, 2),
            'irpf': round(irpf_amount, 2),
            'total': round(total, 2)
        }

    def generate_invoice(self, client_id, invoice_number=None, invoice_date=None):
        """
        Generate a complete invoice for a client
        """
        client_data = self.get_client_data(client_id)
        if client_data is None:
            return None

        client_works = self.get_client_works(client_id)
        if client_works.empty:
            print(f"Warning: No works found for client ID '{client_id}'")

        totals = self.calculate_invoice_totals(client_works)

        if invoice_number is None:
            inp_num = input("Enter invoice number (or press Enter for auto): ").strip()
            invoice_number = inp_num if inp_num else f"INV-{datetime.datetime.now().strftime('%Y%m%d')}-{client_id}"

        if invoice_date is None:
            inp_date = input("Enter invoice date (YYYY-MM-DD or press Enter for today): ").strip()
            invoice_date = inp_date if inp_date else datetime.datetime.now().strftime('%Y-%m-%d')

        invoice = {
            'invoice_number': invoice_number,
            'invoice_date': invoice_date,
            'client_data': client_data,
            'works': client_works.to_dict('records'),
            'subtotal': totals['subtotal'],
            'tax_rate': self.tax_rate * 100,
            'tax_amount': totals['tax'],
            'irpf_rate': self.irpf_rate * 100,
            'irpf_amount': totals['irpf'],
            'total_amount': totals['total']
        }

        return invoice

    def print_invoice(self, invoice):
        """Print invoice in a formatted way"""
        if invoice is None:
            print("No invoice to print")
            return

        line_sep = "=" * 92
        thin_sep = "-" * 92

        print(line_sep)
        print(f"INVOICE #{invoice['invoice_number']}")
        print(line_sep)
        print(f"Date: {invoice['invoice_date']}\n")

        # Client information
        print("BILL TO:")
        client = invoice['client_data']

        name_cols = ['Name', 'name', 'company_name', 'client_name', 'Company_Name', 'Client_Name']
        client_name = next((client[col] for col in name_cols if col in client and pd.notna(client[col])), "Unknown Client")
        print(f"  {client_name}")

        nif_cols = ['NIF_DNI', 'nif', 'NIF', 'cif', 'CIF']
        for col in nif_cols:
            if col in client and pd.notna(client[col]):
                print(f"  NIF/CIF: {client[col]}")
                break

        addr_cols = ['Adress', 'address', 'Address', 'street', 'Street']
        for col in addr_cols:
            if col in client and pd.notna(client[col]):
                print(f"  {client[col]}")
                break

        zip_cols = ['ZIP', 'zip_code', 'zip', 'postal_code', 'Zip_Code', 'Postal_Code']
        city_cols = ['City_Town', 'city', 'City']
        zip_val = next((str(int(client[c])) if isinstance(client[c], float) else str(client[c]) for c in zip_cols if c in client and pd.notna(client[c])), "")
        city_val = next((str(client[c]) for c in city_cols if c in client and pd.notna(client[c])), "")

        if zip_val or city_val:
            print(f"  {zip_val} {city_val}".strip())

        print("\nCONCEPTOS-TRABAJOS:")
        print(thin_sep)
        print(f"{'Description':<70} {'Date':<12} {'Amount':<10}")
        print(thin_sep)

        for work in invoice['works']:
            desc_cols = ['Concept', 'description', 'concept', 'service', 'Description', 'Service']
            description = next((str(work[col]) for col in desc_cols if col in work and pd.notna(work[col])), "Service")

            date_cols = ['Date', 'date', 'work_date', 'Work_Date']
            work_date = "N/A"
            for col in date_cols:
                if col in work and pd.notna(work[col]):
                    work_date = str(work[col])[:10]
                    break

            amount_cols = ['Importe', 'amount', 'import', 'price', 'cost', 'value', 'total', 'Amount']
            amount = next((work[col] for col in amount_cols if col in work and pd.notna(work[col])), 0)

            print(f"  Description: {description}")
            print(f"  Date: {work_date:<12} Amount: Eur {amount:<9.2f}\n")

        print(thin_sep)
        print(f"{'Subtotal (Base Imponible):':<79} Eur {invoice['subtotal']:>8.2f}")
        print(f"{'Tax (' + str(invoice['tax_rate']) + '%):':<79} Eur {invoice['tax_amount']:>8.2f}")
        print(f"{'IRPF (' + str(invoice['irpf_rate']) + '%):':<79} Eur {invoice['irpf_amount']:>8.2f}")
        print(line_sep)
        print(f"{'TOTAL (Including VAT and IRPF):':<79} Eur {invoice['total_amount']:>8.2f}")
        print(line_sep)

    def save_invoice_to_file(self, invoice, filename=None):
        """Save invoice to a text file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            safe_num = str(invoice['invoice_number']).replace('/', '_')
            filename = f"invoice_{safe_num}.txt"

        import sys
        original_stdout = sys.stdout

        try:
            with open(filename, 'w', encoding='utf-8') as f:
                sys.stdout = f
                self.print_invoice(invoice)
            sys.stdout = original_stdout
            print(f"Invoice saved to {filename}")
        except Exception as e:
            sys.stdout = original_stdout
            print(f"Error saving invoice: {e}")

    def save_invoice_to_pdf(self, invoice, filename=None):
        """Save invoice to a PDF file"""
        if invoice is None:
            print("No invoice to save")
            return

        if filename is None:
            safe_num = str(invoice['invoice_number']).replace('/', '_')
            filename = f"invoice_{safe_num}.pdf"

        try:
            doc = SimpleDocTemplate(filename, pagesize=A4,
                                    rightMargin=50, leftMargin=50,
                                    topMargin=50, bottomMargin=50)

            styles = getSampleStyleSheet()

            title_style = ParagraphStyle(
                'InvoiceTitle',
                parent=styles['Heading1'],
                fontSize=22,
                spaceAfter=15,
                textColor=colors.darkblue,
                alignment=TA_CENTER,
                fontName='Helvetica-Bold'
            )

            header_style = ParagraphStyle(
                'InvoiceHeader',
                parent=styles['Heading2'],
                fontSize=12,
                spaceAfter=8,
                spaceBefore=14,
                textColor=colors.darkblue,
                fontName='Helvetica-Bold'
            )

            normal_style = ParagraphStyle(
                'InvoiceNormal',
                parent=styles['Normal'],
                fontSize=9,
                spaceAfter=4,
                leading=12,
                fontName='Helvetica'
            )

            client_style = ParagraphStyle(
                'ClientInfo',
                parent=styles['Normal'],
                fontSize=10,
                spaceAfter=4,
                leading=14,
                fontName='Helvetica-Bold'
            )

            story = []

            # Encabezado Emisor
            story.append(Paragraph("Factura", title_style))
            story.append(Paragraph("Servacal_SPV Servicios Veterinarios", normal_style))
            story.append(Paragraph("Rafael Vazquez Caldito. 08847362K", normal_style))
            story.append(Paragraph("Enrique Segura Otano, 5. 7A.", normal_style))
            story.append(Paragraph("06004. Badajoz (Badajoz).", normal_style))
            story.append(Spacer(1, 10))

            # Datos de Factura
            story.append(Paragraph(f"<b>Factura N_#:</b> {str(invoice['invoice_number'])}", normal_style))
            story.append(Paragraph(f"<b>Fecha:</b> {str(invoice['invoice_date'])}", normal_style))
            story.append(Spacer(1, 10))

            # Datos del Cliente
            story.append(Paragraph("Facturado a:", header_style))
            client = invoice['client_data']

            name_cols = ['Name', 'name', 'company_name', 'client_name', 'Company_Name', 'Client_Name']
            client_name = next((client[col] for col in name_cols if col in client and pd.notna(client[col])), "Unknown Client")
            story.append(Paragraph(str(client_name), client_style))

            nif_cols = ['NIF_DNI', 'nif', 'NIF', 'cif', 'CIF']
            for col in nif_cols:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(f"NIF/CIF: {client[col]}", normal_style))
                    break

            addr_cols = ['Adress', 'address', 'Address', 'street', 'Street']
            for col in addr_cols:
                if col in client and pd.notna(client[col]):
                    story.append(Paragraph(str(client[col]), normal_style))
                    break

            zip_cols = ['ZIP', 'zip_code', 'zip', 'postal_code', 'Zip_Code', 'Postal_Code']
            city_cols = ['City_Town', 'city', 'City']
            zip_val = next((str(int(client[c])) if isinstance(client[c], float) else str(client[c]) for c in zip_cols if c in client and pd.notna(client[c])), "")
            city_val = next((str(client[c]) for c in city_cols if c in client and pd.notna(client[c])), "")

            if zip_val or city_val:
                story.append(Paragraph(f"{zip_val} {city_val}".strip(), normal_style))

            story.append(Spacer(1, 10))

            # Tabla de Conceptos
            story.append(Paragraph("CONCEPTOS-TRABAJOS:", header_style))
            story.append(Spacer(1, 5))

            table_data = [['Descripcion', 'Fecha', 'Importe']]

            for work in invoice['works']:
                desc_cols = ['Concept', 'description', 'concept', 'service', 'Description', 'Service']
                description = next((str(work[col]) for col in desc_cols if col in work and pd.notna(work[col])), "Service")

                date_cols = ['Date', 'date', 'work_date', 'Work_Date']
                work_date = "N/A"
                for col in date_cols:
                    if col in work and pd.notna(work[col]):
                        work_date = str(work[col])[:10]
                        break

                amount_cols = ['Importe', 'amount', 'import', 'price', 'cost', 'value', 'total', 'Amount']
                amount = next((work[col] for col in amount_cols if col in work and pd.notna(work[col])), 0)

                table_data.append([Paragraph(description, normal_style), work_date, f"Eur {amount:.2f}"])

            # Filas de Totales
            table_data.append(['', '', ''])
            table_data.append(['Subtotal (Base Imponible):', '', f"Eur {invoice['subtotal']:.2f}"])
            table_data.append([f'IVA ({invoice["tax_rate"]:.1f}%):', '', f"Eur {invoice['tax_amount']:.2f}"])
            table_data.append([f'IRPF ({invoice["irpf_rate"]:.1f}%):', '', f"Eur {invoice['irpf_amount']:.2f}"])
            table_data.append(['TOTAL:', '', f"Eur {invoice['total_amount']:.2f}"])

            table = Table(table_data, colWidths=['*', 1.2*inch, 1.2*inch])

            LAST_ROW = -1
            SUBTOTAL_ROW = -4
            BLANK_ROW = -5

            table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('ALIGN', (2, 0), (2, -1), 'RIGHT'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, 0), 10),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 8),

                ('FONTNAME', (0, 1), (-1, BLANK_ROW), 'Helvetica'),
                ('FONTSIZE', (0, 1), (-1, BLANK_ROW), 9),
                ('GRID', (0, 0), (-1, BLANK_ROW), 0.5, colors.grey),

                ('FONTNAME', (0, SUBTOTAL_ROW), (-1, LAST_ROW), 'Helvetica-Bold'),
                ('FONTSIZE', (0, SUBTOTAL_ROW), (-1, LAST_ROW), 10),

                ('LINEABOVE', (0, LAST_ROW), (-1, LAST_ROW), 1.5, colors.black),
                ('BACKGROUND', (0, LAST_ROW), (-1, LAST_ROW), colors.lightgrey),
            ]))

            story.append(table)
            story.append(Spacer(1, 20))

            # Pie de página / Datos Bancarios
            story.append(Paragraph("Cuenta Corriente de abono, numero:", normal_style))
            story.append(Paragraph("ES34 3009 0091 8123 2755 9213", client_style))

            doc.build(story)
            print(f"Invoice PDF saved to {filename}")

        except Exception as e:
            print(f"Error saving invoice to PDF: {e}")


def main():
    generator = InvoiceGenerator(
        clients_csv_path="clients.csv",
        works_csv_path="works.csv",
        tax_rate=0.10,   # 21% IVA or 10%
        irpf_rate=0.00   # 15% IRPF
    )

    try:
        client_id_input = input("Enter ID_client: ").strip()
        client_id = int(client_id_input) if client_id_input.isdigit() else client_id_input
    except ValueError:
        client_id = client_id_input

    invoice = generator.generate_invoice(client_id)

    if invoice:
        generator.print_invoice(invoice)
        generator.save_invoice_to_file(invoice)
        generator.save_invoice_to_pdf(invoice)
    else:
        print("Failed to generate invoice")

if __name__ == "__main__":
    main()

Loaded 460 clients from clients.csv
Loaded 9 work records from works.csv
Enter ID_client: 3
Enter invoice number (or press Enter for auto): B2026-807
Enter invoice date (YYYY-MM-DD or press Enter for today): 2026-10-01
INVOICE #B2026-807
Date: 2026-10-01

BILL TO:
  Concepcion Arranz Gilarranz
  NIF/CIF: 02822282K
  Raimundo Fernandez Villaverde, 26; apartamento 352.
28003 Madrid

CONCEPTOS-TRABAJOS:
--------------------------------------------------------------------------------------------
Description                                                            Date         Amount    
--------------------------------------------------------------------------------------------
  Description: Desplazamiento a Sierra de la Cruz para la vacunacion de todas las enfermedades a 13 becerros. Enfermedades que afectan a cebo. Tiempo y mano de obra. Zoosanitarios empleados.
  Date: 2026-09-01   Amount: Eur 31.59    

  Description: Dejada en Valverde de leganes los dos envases para la desparasita